In [ ]:
# Cell 0: Setup — install packages, mount Drive
!pip install -q torch transformers==5.6.2 datasets sentencepiece seqeval pyarrow huggingface_hub
from google.colab import drive
drive.mount('/content/drive')
import torch, transformers
print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 102.6 MB/s eta 0:00:00
Mounted at /content/drive
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, VRAM: 102.0 GB
transformers: 5.6.2
torch: 2.10.0+cu128


In [ ]:
# Cell 1: Clone repo + download prepared training data (we'll need this for Stage 2 gold + dev)
import os, subprocess
if not os.path.exists("/content/kniv-nlp-models"):
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/rustic-ai/kniv-nlp-models.git"], cwd="/content")
from huggingface_hub import snapshot_download
snapshot_download("dragonscale-ai/kniv-corpus-en", repo_type="dataset",
                  allow_patterns="prepared/kniv-deberta-cascade/*", local_dir="/content/kniv-data")
os.chdir("/content/kniv-nlp-models")
os.makedirs("data/prepared", exist_ok=True)
if not os.path.exists("data/prepared/kniv-deberta-cascade"):
    os.symlink("/content/kniv-data/prepared/kniv-deberta-cascade", "data/prepared/kniv-deberta-cascade")
print(f"Working dir: {os.getcwd()}")
print(f"Data ready: {os.path.exists('data/prepared/kniv-deberta-cascade')}")

Fetching 40 files:   0%|          | 0/40 [00:00<?, ?it/s]

Working dir: /content/kniv-nlp-models
Data ready: True


In [ ]:
# Cell 2: Imports, label sets, and shared model components

import json, random, re, os, shutil, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

device = torch.device("cuda")
DATA = "data/prepared/kniv-deberta-cascade"
DISTILL_DIR = "/content/drive/MyDrive/kniv-models/distillation_v2"
STUDENT_DIR = "/content/drive/MyDrive/kniv-models/students_v2"
os.makedirs(STUDENT_DIR, exist_ok=True)

# Load distillation metadata
with open(f"{DISTILL_DIR}/metadata.json") as f:
    DISTILL_META = json.load(f)
print(f"Distillation v2: {DISTILL_META['total_sentences']:,} sentences, "
      f"{DISTILL_META['total_size_gb']:.1f} GB across {DISTILL_META['total_shards']} shards")
TEACHER_LAYERS = DISTILL_META["teacher_layers_stored"]
print(f"  Teacher layers stored: {TEACHER_LAYERS}")

# ── Label sets ──────────────────────────────────────────
POS_LABELS = ["ADJ","ADP","ADV","AUX","CCONJ","DET","INTJ","NOUN","NUM","PART","PRON","PROPN","PUNCT","SCONJ","SYM","VERB","X"]
NER_LABELS = ["O","B-PERSON","I-PERSON","B-NORP","I-NORP","B-FAC","I-FAC","B-ORG","I-ORG","B-GPE","I-GPE","B-LOC","I-LOC","B-PRODUCT","I-PRODUCT","B-EVENT","I-EVENT","B-WORK_OF_ART","I-WORK_OF_ART","B-LAW","I-LAW","B-LANGUAGE","I-LANGUAGE","B-DATE","I-DATE","B-TIME","I-TIME","B-PERCENT","I-PERCENT","B-MONEY","I-MONEY","B-QUANTITY","I-QUANTITY","B-ORDINAL","I-ORDINAL","B-CARDINAL","I-CARDINAL"]
SRL_TAGS = ["O","V","B-ARG0","I-ARG0","B-ARG1","I-ARG1","B-ARG2","I-ARG2","B-ARG3","I-ARG3","B-ARG4","I-ARG4","B-ARGM-TMP","I-ARGM-TMP","B-ARGM-LOC","I-ARGM-LOC","B-ARGM-MNR","I-ARGM-MNR","B-ARGM-CAU","I-ARGM-CAU","B-ARGM-PRP","I-ARGM-PRP","B-ARGM-NEG","I-ARGM-NEG","B-ARGM-ADV","I-ARGM-ADV","B-ARGM-DIR","I-ARGM-DIR","B-ARGM-DIS","I-ARGM-DIS","B-ARGM-EXT","I-ARGM-EXT","B-ARGM-MOD","I-ARGM-MOD","B-ARGM-PRD","I-ARGM-PRD","B-ARGM-GOL","I-ARGM-GOL","B-ARGM-COM","I-ARGM-COM","B-ARGM-REC","I-ARGM-REC"]
CLS_LABELS = ["inform","request","question","confirm","reject","offer","social","status"]
DEPREL_LIST = ["root","acl","acl:relcl","advcl","advcl:relcl","advmod","amod","appos","aux","aux:pass","case","cc","cc:preconj","ccomp","compound","compound:prt","conj","cop","csubj","csubj:outer","csubj:pass","dep","det","det:predet","discourse","dislocated","expl","fixed","flat","goeswith","iobj","list","mark","nmod","nmod:desc","nmod:npmod","nmod:poss","nmod:tmod","nsubj","nsubj:outer","nsubj:pass","nummod","obj","obl","obl:agent","obl:npmod","obl:tmod","orphan","parataxis","punct","reparandum","vocative","xcomp"]
N_POS, N_NER, N_SRL, N_DEP, N_CLS = len(POS_LABELS), len(NER_LABELS), len(SRL_TAGS), len(DEPREL_LIST), len(CLS_LABELS)
H_TEACHER = 1024  # teacher's hidden dim
pos_map = {l: i for i, l in enumerate(POS_LABELS)}
ner_map = {l: i for i, l in enumerate(NER_LABELS)}
srl_map = {l: i for i, l in enumerate(SRL_TAGS)}
cls_map = {l: i for i, l in enumerate(CLS_LABELS)}

# ── Shared model components ─────────────────────────────
class ScalarMix(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.weights = nn.Parameter(torch.zeros(n))
        self.scale = nn.Parameter(torch.ones(1))
    def forward(self, layers):
        w = torch.softmax(self.weights, dim=0)
        return self.scale * sum(wi * li for wi, li in zip(w, layers))

class AttentionPool(nn.Module):
    def __init__(self, H):
        super().__init__()
        self.attn = nn.Linear(H, 1)
    def forward(self, hidden, mask):
        scores = self.attn(hidden).squeeze(-1).masked_fill(~mask.bool(), -1e9)
        return (hidden * torch.softmax(scores, -1).unsqueeze(-1)).sum(1)

class Biaffine(nn.Module):
    def __init__(self, in_dim, out_dim=1):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(out_dim, in_dim+1, in_dim+1))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, h_dep, h_head):
        B, S, D = h_dep.size()
        h_dep = torch.cat([h_dep, torch.ones(B,S,1,device=h_dep.device)], -1)
        h_head = torch.cat([h_head, torch.ones(B,S,1,device=h_head.device)], -1)
        scores = torch.einsum("bxi,oij,byj->boxy", h_dep, self.weight, h_head)
        return scores.squeeze(1).contiguous() if scores.size(1)==1 else scores.permute(0,2,3,1).contiguous()

class BiaffineDEPHead(nn.Module):
    def __init__(self, H, arc_dim, label_dim, num_labels, dropout=0.1):
        super().__init__()
        self.arc_dep = nn.Sequential(nn.Linear(H, arc_dim), nn.LayerNorm(arc_dim), nn.GELU(), nn.Dropout(dropout))
        self.arc_head = nn.Sequential(nn.Linear(H, arc_dim), nn.LayerNorm(arc_dim), nn.GELU(), nn.Dropout(dropout))
        self.label_dep = nn.Sequential(nn.Linear(H, label_dim), nn.LayerNorm(label_dim), nn.GELU(), nn.Dropout(dropout))
        self.label_head = nn.Sequential(nn.Linear(H, label_dim), nn.LayerNorm(label_dim), nn.GELU(), nn.Dropout(dropout))
        self.biaffine_arc = Biaffine(arc_dim, 1)
        self.biaffine_label = Biaffine(label_dim, num_labels)
    def forward(self, hidden):
        return (self.biaffine_arc(self.arc_dep(hidden), self.arc_head(hidden)),
                self.biaffine_label(self.label_dep(hidden), self.label_head(hidden)))

print(f"Labels: POS={N_POS}, NER={N_NER}, SRL={N_SRL}, DEP={N_DEP}, CLS={N_CLS}")
print(f"Teacher hidden dim: {H_TEACHER}")
print("Setup complete.")

Distillation v2: 206,520 sentences, 56.6 GB across 5 shards
  Teacher layers stored: [12, 18, 24]
Labels: POS=17, NER=37, SRL=42, DEP=53, CLS=8
Teacher hidden dim: 1024
Setup complete.


In [ ]:
# Cell 3: StudentCascadeV5 — proven baseline architecture (no SRL-specific changes)
# Identical to v5 from previous notebooks. Auto-scales to any encoder hidden dim.

class StudentCascadeV5(nn.Module):
    """v5: DEP cascade into SRL + wider BiLSTM for span tracking. Champion architecture.
    Cascade: POS → NER → DEP → SRL (each detached, providing soft features downstream)
    """
    def __init__(self, encoder, H, num_layers):
        super().__init__()
        self.encoder = encoder
        self.pred_embedding = nn.Embedding(2, H)
        nn.init.zeros_(self.pred_embedding.weight)
        NL = num_layers

        # POS
        self.pos_sm = ScalarMix(NL)
        self.pos_head = nn.Linear(H, N_POS)

        # NER
        self.ner_sm = ScalarMix(NL)
        lstm_h = max(H // 4, 64)
        self.ner_lstm = nn.LSTM(H, lstm_h, bidirectional=True, batch_first=True)
        self.ner_proj = nn.Linear(lstm_h * 2, H)
        self.ner_head = nn.Sequential(nn.LayerNorm(H+N_POS), nn.Linear(H+N_POS, H),
                                      nn.GELU(), nn.Dropout(0.1), nn.Linear(H, N_NER))

        # DEP — BiLSTM before biaffine
        self.dep_sm = ScalarMix(NL)
        in_dim = H + N_POS + N_NER
        self.dep_proj = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, H), nn.GELU())
        dep_lstm_h = max(H // 4, 64)
        self.dep_lstm = nn.LSTM(H, dep_lstm_h, bidirectional=True, batch_first=True)
        self.dep_lstm_proj = nn.Linear(dep_lstm_h * 2, H)
        arc_dim, label_dim = max(H//2, 64), max(H//8, 32)
        self.dep_biaff = BiaffineDEPHead(H, arc_dim, label_dim, N_DEP)

        # SRL — predicate interaction + DEP cascade + POS cascade + wider BiLSTM
        srl_input_dim = H * 4 + N_POS + N_DEP
        self.srl_sm = ScalarMix(NL)
        self.srl_interaction_proj = nn.Sequential(
            nn.LayerNorm(srl_input_dim),
            nn.Linear(srl_input_dim, H), nn.GELU(), nn.Dropout(0.1))
        srl_lstm_h = max(H // 2, 96)
        self.srl_lstm = nn.LSTM(H, srl_lstm_h, bidirectional=True, batch_first=True)
        self.srl_proj = nn.Linear(srl_lstm_h * 2, H)
        self.srl_head = nn.Sequential(nn.Dropout(0.1), nn.Linear(H, H), nn.GELU(),
                                      nn.Dropout(0.1), nn.Linear(H, N_SRL))

        # CLS
        self.cls_sm = ScalarMix(NL)
        self.cls_pool = AttentionPool(H)
        self.cls_head = nn.Sequential(nn.LayerNorm(H), nn.Linear(H, H * 2), nn.GELU(),
                                      nn.Dropout(0.1), nn.Linear(H * 2, H), nn.GELU(),
                                      nn.Dropout(0.1), nn.Linear(H, N_CLS))

        # ── NEW v2: Hidden state distillation projections ──
        # Project student hidden states up to teacher dim (1024) for MSE loss
        # One projection per teacher layer we track
        # These are train-only — not needed for inference
        self.distill_projections = nn.ModuleDict({
            f"l{l}": nn.Linear(H, H_TEACHER, bias=False) for l in TEACHER_LAYERS
        })
        # Initialize as identity-like (small random) for stable training
        for proj in self.distill_projections.values():
            nn.init.xavier_uniform_(proj.weight, gain=0.5)

    def forward(self, input_ids, attention_mask, predicate_idx, return_hidden_states=False):
        B, S = input_ids.size()
        if hasattr(self.encoder, 'embeddings'):
            emb = self.encoder.embeddings(input_ids)
            indicator = torch.zeros(B, S, dtype=torch.long, device=input_ids.device)
            indicator.scatter_(1, predicate_idx.unsqueeze(1), 1)
            emb = emb + self.pred_embedding(indicator)
            enc_out = self.encoder.encoder(emb, attention_mask, output_hidden_states=True)
        else:
            enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                                   output_hidden_states=True)
        layers = list(enc_out.hidden_states)

        # POS
        pos_logits = self.pos_head(self.pos_sm(layers))
        pos_p = torch.softmax(pos_logits, -1)

        # NER
        ner_h = self.ner_sm(layers)
        lo, _ = self.ner_lstm(ner_h)
        ner_logits = self.ner_head(torch.cat([self.ner_proj(lo) + ner_h, pos_p.detach()], -1))
        ner_p = torch.softmax(ner_logits, -1)

        # DEP (BiLSTM + biaffine)
        dep_h = self.dep_sm(layers)
        dep_proj = self.dep_proj(torch.cat([dep_h, pos_p.detach(), ner_p.detach()], -1))
        dep_lo, _ = self.dep_lstm(dep_proj)
        dep_adapted = self.dep_lstm_proj(dep_lo) + dep_proj
        arc_scores, label_scores = self.dep_biaff(dep_adapted)

        # DEP cascade features for SRL
        pred_heads = arc_scores.argmax(dim=-1)
        ph = pred_heads.clamp(min=0).unsqueeze(-1).unsqueeze(-1).expand(B, S, 1, N_DEP)
        dep_label_at_head = label_scores.gather(2, ph).squeeze(2)
        dep_rel_probs = torch.softmax(dep_label_at_head, dim=-1).detach()

        # SRL
        srl_h = self.srl_sm(layers)
        pred_idx_expanded = predicate_idx.view(B, 1, 1).expand(B, 1, srl_h.size(-1))
        h_pred = srl_h.gather(1, pred_idx_expanded).expand_as(srl_h)
        h_i = srl_h
        srl_features = torch.cat([
            h_i, h_pred, h_i * h_pred, (h_i - h_pred).abs(),
            pos_p.detach(), dep_rel_probs,
        ], dim=-1)
        srl_projected = self.srl_interaction_proj(srl_features)
        srl_lo, _ = self.srl_lstm(srl_projected)
        srl_adapted = self.srl_proj(srl_lo) + srl_projected
        srl_logits = self.srl_head(srl_adapted)

        # CLS
        cls_h = self.cls_sm(layers)
        cls_logits = self.cls_head(self.cls_pool(cls_h, attention_mask))

        if return_hidden_states:
            # Map student layers to teacher's tracked layers
            # Student has variable layers; we map proportionally to teacher's [12, 18, 24]
            # E.g., student_layers=12 → mapped to {6, 9, 12}; student_layers=6 → {3, 4, 6}
            n_student_layers = len(layers) - 1  # skip embedding layer
            mapped_hidden = {}
            for tl in TEACHER_LAYERS:
                # Map teacher layer 12/18/24 (out of 24) to corresponding student layer
                student_layer_idx = max(1, round(n_student_layers * tl / 24))
                mapped_hidden[tl] = layers[student_layer_idx]  # skip embedding (idx 0)
            return pos_logits, ner_logits, arc_scores, label_scores, srl_logits, cls_logits, mapped_hidden
        return pos_logits, ner_logits, arc_scores, label_scores, srl_logits, cls_logits


# Viterbi decode for SRL eval
def viterbi_decode_srl(logits):
    """Constrained BIO Viterbi for SRL tags."""
    num_tags, seq_len = logits.size(-1), logits.size(0)
    lp = torch.log_softmax(logits, dim=-1)
    allowed = torch.ones(num_tags, num_tags, dtype=torch.bool)
    tag_names = {i: t for i, t in enumerate(SRL_TAGS)}
    tag_ids = {t: i for i, t in enumerate(SRL_TAGS)}
    for j in range(num_tags):
        if tag_names[j].startswith("I-"):
            role = tag_names[j][2:]
            b_tag = tag_ids.get(f"B-{role}", -1)
            for i in range(num_tags):
                if i != b_tag and i != j: allowed[i][j] = False
    v = torch.full((seq_len, num_tags), -1e9); bp = torch.zeros(seq_len, num_tags, dtype=torch.long)
    v[0] = lp[0]
    for t in range(1, seq_len):
        for j in range(num_tags):
            s = v[t-1].clone(); s[~allowed[:, j]] = -1e9
            b = s.argmax(); v[t, j] = s[b] + lp[t, j]; bp[t, j] = b
    path = [0] * seq_len; path[-1] = v[-1].argmax().item()
    for t in range(seq_len - 2, -1, -1): path[t] = bp[t+1, path[t+1]].item()
    return path


print("StudentCascadeV5 + Viterbi defined.")
print(f"  Hidden state distillation projections: {len(TEACHER_LAYERS)} (one per teacher layer {TEACHER_LAYERS})")

StudentCascadeV5 + Viterbi defined.
  Hidden state distillation projections: 3 (one per teacher layer [12, 18, 24])


In [ ]:
# Cell 4: DistillationDatasetV2 — reads enriched parquet (per-verb data, hidden states)

class DistillationDatasetV2(Dataset):
    """Reads v2 distillation data with per-verb sampling for SRL.

    Each __getitem__ returns:
        - Sentence-level: input_ids, attention_mask, valid mask, words
        - Soft targets: pos_t, ner_t, cls_t, srl_t (sampled-verb), dep_*
        - Hard targets: pos_h, ner_h, srl_hard (sampled-verb), dep_head_hard, dep_rel_hard
        - Predicate: predicate_idx (sampled verb's token position)
        - Hidden states: teacher_hidden_l12, l18, l24 (for hidden state distillation)
        - Predicate-aware hidden: teacher_pred_hidden_l24 (for sampled verb)
        - Validity flags: srl_valid (False if no verbs in sentence)
    """
    def __init__(self, parquet_paths, student_tokenizer, max_length=128):
        self.tok = student_tokenizer
        self.ml = max_length
        tables = [pq.read_table(p) for p in parquet_paths]
        self.data = pa.concat_tables(tables).to_pandas()
        print(f"  Loaded {len(self.data):,} examples from {len(parquet_paths)} shards")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        words = list(row["words"])
        n = int(row["n_words"])
        S = self.ml

        # Tokenize for student
        enc = self.tok(words, is_split_into_words=True, max_length=S,
                       padding="max_length", truncation=True, return_tensors="pt")
        word_ids = enc.word_ids()

        # Build word→token index map
        w2t = {}; prev = None
        for k, wid in enumerate(word_ids):
            if wid is not None and wid != prev and wid < n: w2t[wid] = k
            prev = wid
        valid_word_indices = sorted(w2t.keys())

        # ── Decode teacher artifacts (sentence-level) ──
        t_pos = np.frombuffer(row["pos_logits"], dtype=np.float16).reshape(n, N_POS).astype(np.float32)
        t_ner = np.frombuffer(row["ner_logits"], dtype=np.float16).reshape(n, N_NER).astype(np.float32)
        t_cls = np.frombuffer(row["cls_logits"], dtype=np.float16).reshape(N_CLS).astype(np.float32)
        t_arc = np.frombuffer(row["dep_arc_scores"], dtype=np.float16).reshape(n, n).astype(np.float32)
        dep_label_top5 = json.loads(row["dep_label_top5"])

        # Teacher hidden states (word-level)
        teacher_hidden = {}
        for tl in TEACHER_LAYERS:
            arr = np.frombuffer(row[f"hidden_l{tl}"], dtype=np.float16).reshape(n, H_TEACHER).astype(np.float32)
            teacher_hidden[tl] = arr

        # ── Per-verb data — sample one verb for SRL distillation this epoch ──
        per_verb_count = int(row["per_verb_count"])
        per_verb_indices = json.loads(row["per_verb_indices"])
        srl_nonzero = per_verb_count > 0

        if srl_nonzero:
            # Random verb pick (stochastic SRL distillation across epochs)
            verb_pick_idx = random.randint(0, per_verb_count - 1)
            verb_idx = per_verb_indices[verb_pick_idx]
            # Decode that verb's per-token SRL logits + predicate-aware hidden state
            srl_size_per_verb = n * N_SRL * 2  # bytes (fp16)
            hidden_size_per_verb = n * H_TEACHER * 2
            srl_offset = verb_pick_idx * srl_size_per_verb
            hidden_offset = verb_pick_idx * hidden_size_per_verb
            t_srl = np.frombuffer(
                row["per_verb_srl_logits"], dtype=np.float16,
                count=n*N_SRL, offset=srl_offset
            ).reshape(n, N_SRL).astype(np.float32)
            t_pred_hidden = np.frombuffer(
                row["per_verb_pred_hidden"], dtype=np.float16,
                count=n*H_TEACHER, offset=hidden_offset
            ).reshape(n, H_TEACHER).astype(np.float32)
            srl_hard_seq = json.loads(row["per_verb"])[verb_pick_idx]["srl_hard"]
        else:
            t_srl = np.zeros((n, N_SRL), dtype=np.float32)
            t_pred_hidden = np.zeros((n, H_TEACHER), dtype=np.float32)
            srl_hard_seq = ["O"] * n
            verb_idx = 0

        # ── Build student-aligned tensors ──
        pos_t = torch.zeros(S, N_POS); ner_t = torch.zeros(S, N_NER); srl_t = torch.zeros(S, N_SRL)
        valid = torch.zeros(S, dtype=torch.bool)
        pos_h = torch.full((S,), -100, dtype=torch.long)
        ner_h = torch.full((S,), -100, dtype=torch.long)
        dep_head_hard = torch.full((S,), -100, dtype=torch.long)
        dep_rel_hard = torch.full((S,), -100, dtype=torch.long)
        srl_hard = torch.full((S,), -100, dtype=torch.long)
        pos_hard_list = list(row["pos_hard"])
        ner_hard_list = list(row["ner_hard"])

        # Hidden state tensors (S, H_TEACHER)
        hidden_targets = {tl: torch.zeros(S, H_TEACHER) for tl in TEACHER_LAYERS}
        pred_hidden_target = torch.zeros(S, H_TEACHER)

        for wid, k in w2t.items():
            pos_t[k] = torch.from_numpy(t_pos[wid])
            ner_t[k] = torch.from_numpy(t_ner[wid])
            srl_t[k] = torch.from_numpy(t_srl[wid])
            valid[k] = True
            pos_h[k] = pos_map.get(pos_hard_list[wid], 0) if wid < len(pos_hard_list) else 0
            ner_h[k] = ner_map.get(ner_hard_list[wid], 0) if wid < len(ner_hard_list) else 0

            # Hidden states
            for tl in TEACHER_LAYERS:
                hidden_targets[tl][k] = torch.from_numpy(teacher_hidden[tl][wid])
            pred_hidden_target[k] = torch.from_numpy(t_pred_hidden[wid])

            # SRL hard label
            if srl_nonzero and wid < len(srl_hard_seq):
                srl_hard[k] = srl_map.get(srl_hard_seq[wid], 0)

            # DEP hard head + relation
            if valid_word_indices:
                arc_scores_valid = [float(t_arc[wid, wj]) for wj in valid_word_indices]
                best_word_idx = valid_word_indices[int(np.argmax(arc_scores_valid))]
                dep_head_hard[k] = w2t[best_word_idx]
            if wid < len(dep_label_top5) and dep_label_top5[wid]:
                dep_rel_hard[k] = int(dep_label_top5[wid][0][0])

        cls_t_tensor = torch.from_numpy(t_cls)
        pred_tidx = w2t.get(verb_idx, 0)

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "predicate_idx": torch.tensor(pred_tidx, dtype=torch.long),
            "valid": valid,
            # Soft targets
            "pos_t": pos_t, "ner_t": ner_t, "srl_t": srl_t, "cls_t": cls_t_tensor,
            # Hard targets
            "pos_h": pos_h, "ner_h": ner_h, "srl_hard": srl_hard,
            "dep_head_hard": dep_head_hard, "dep_rel_hard": dep_rel_hard,
            "srl_valid": torch.tensor(srl_nonzero, dtype=torch.bool),
            # NEW v2: hidden state distillation targets
            **{f"teacher_hidden_l{tl}": hidden_targets[tl] for tl in TEACHER_LAYERS},
            "teacher_pred_hidden": pred_hidden_target,
        }


def collate(feats):
    return {k: torch.stack([f[k] for f in feats]) for k in feats[0]}


print("DistillationDatasetV2 defined.")
print("  Per-verb sampling: each __getitem__ randomly picks one verb's SRL data")
print(f"  Hidden state targets: {[f'teacher_hidden_l{tl}' for tl in TEACHER_LAYERS]}")

DistillationDatasetV2 defined.
  Per-verb sampling: each __getitem__ randomly picks one verb's SRL data
  Hidden state targets: ['teacher_hidden_l12', 'teacher_hidden_l18', 'teacher_hidden_l24']


In [ ]:
# Cell 5: Loss functions for v2 distillation
# Tier-1 distillation losses: logit KL + hard CE + hidden state MSE + R-Drop

def kl_loss(student_logits, teacher_logits, valid_mask, temperature=3.0):
    """KL divergence on softened predictions for valid token positions."""
    sl = student_logits.view(-1, student_logits.size(-1))[valid_mask]
    tl = teacher_logits.view(-1, teacher_logits.size(-1))[valid_mask]
    if sl.numel() == 0:
        return torch.tensor(0.0, device=student_logits.device)
    return F.kl_div(F.log_softmax(sl/temperature, -1),
                    F.softmax(tl/temperature, -1),
                    reduction="batchmean") * temperature**2


def hard_ce(student_logits, hard_labels):
    """Standard cross-entropy with -100 ignore index."""
    return F.cross_entropy(student_logits.view(-1, student_logits.size(-1)),
                           hard_labels.view(-1), ignore_index=-100)


def hidden_state_mse(student_hidden_proj, teacher_hidden, valid_mask):
    """MSE between projected student hidden state and teacher hidden state.
    Both are LayerNorm'd first for scale invariance.
    Args:
        student_hidden_proj: [B, S, H_TEACHER] — student hidden after up-projection
        teacher_hidden: [B, S, H_TEACHER]
        valid_mask: [B*S] bool
    """
    sh = student_hidden_proj.view(-1, H_TEACHER)[valid_mask]
    th = teacher_hidden.view(-1, H_TEACHER)[valid_mask]
    if sh.numel() == 0:
        return torch.tensor(0.0, device=student_hidden_proj.device)
    # Layer-norm both for stable training (TinyBERT/PKD pattern)
    sh = F.layer_norm(sh, [H_TEACHER])
    th = F.layer_norm(th, [H_TEACHER])
    return F.mse_loss(sh, th)


def rdrop_kl(logits1, logits2, valid_mask, temperature=1.0):
    """Symmetric KL between two student forward passes (R-Drop).
    Encourages consistency across dropout masks. Both directions averaged.
    """
    l1 = logits1.view(-1, logits1.size(-1))[valid_mask]
    l2 = logits2.view(-1, logits2.size(-1))[valid_mask]
    if l1.numel() == 0:
        return torch.tensor(0.0, device=logits1.device)
    p1 = F.log_softmax(l1/temperature, -1)
    p2 = F.log_softmax(l2/temperature, -1)
    kl_12 = F.kl_div(p1, p2, log_target=True, reduction="batchmean")
    kl_21 = F.kl_div(p2, p1, log_target=True, reduction="batchmean")
    return 0.5 * (kl_12 + kl_21) * temperature**2


print("Distillation losses defined:")
print("  kl_loss(s, t, mask, T=3.0)        — soft logit distillation")
print("  hard_ce(s, hard_labels)            — hard label CE")
print("  hidden_state_mse(s_proj, t, mask)  — TinyBERT/PKD-style hidden alignment")
print("  rdrop_kl(s1, s2, mask)             — consistency regularization (R-Drop)")

Distillation losses defined:
  kl_loss(s, t, mask, T=3.0)        — soft logit distillation
  hard_ce(s, hard_labels)            — hard label CE
  hidden_state_mse(s_proj, t, mask)  — TinyBERT/PKD-style hidden alignment
  rdrop_kl(s1, s2, mask)             — consistency regularization (R-Drop)


In [ ]:
# Cell 6: Training pipeline — generic for any student size

def train_student(STUDENT_NAME, suffix="v2",
                  EPOCHS_S1=8, EPOCHS_S2A=8, EPOCHS_S2B=4,
                  BATCH_SIZE=64,
                  LR_ENC=2e-5, LR_HEADS=1e-3,
                  LR_S2_ENC=3e-6, LR_S2_HEADS=1e-3,
                  S1_WARMUP=0.06,
                  USE_RDROP=True, USE_HIDDEN_DISTILL=True,
                  loss_weights=None):
    """Train a student with v2 distillation pipeline.

    Stage 1: 4-loss distillation (logit KL + hard CE + hidden MSE + R-Drop) — all heads, joint
    Stage 2a: SRL gold + teacher silver fine-tune (encoder + SRL components only)
    Stage 2b: CLS-only fine-tune (frozen encoder + SRL)

    Note: we already saw v9's frozen-encoder Stage 2b hurts CLS more than helps SRL,
    so for v2 baseline, run only Stage 1 + Stage 2a (joint). Stage 2b is opt-in for ablations.
    """
    if loss_weights is None:
        loss_weights = {
            "alpha_hard": 0.5,    # weight for hard CE (POS, NER)
            "beta_kl": 1.0,       # weight for soft KL (POS, NER, SRL, CLS)
            "gamma_hidden": 0.3,  # weight for hidden state MSE (each layer separately)
            "delta_rdrop": 0.2,   # weight for R-Drop KL (POS, NER, SRL)
            "dep_arc": 1.5, "dep_rel": 0.5,  # DEP arc/relation hard CE weights
            "ner_mult": 1.5,      # NER importance multiplier
            "srl_mult": 1.0,
            "cls_mult": 1.0,
            "hidden_mult_pred": 0.5,  # weight for predicate-aware hidden state MSE
        }

    print("=" * 70)
    print(f"TRAINING {STUDENT_NAME} v2 ({suffix})")
    print(f"  Stage 1: {EPOCHS_S1} epochs distillation")
    print(f"  Stage 2a: {EPOCHS_S2A} epochs SRL gold+silver fine-tune")
    print(f"  R-Drop: {USE_RDROP}, Hidden distillation: {USE_HIDDEN_DISTILL}")
    print("=" * 70)

    student_tok = AutoTokenizer.from_pretrained(STUDENT_NAME)
    student_enc = AutoModel.from_pretrained(STUDENT_NAME).float()
    H = student_enc.config.hidden_size
    NL = student_enc.config.num_hidden_layers + 1

    model = StudentCascadeV5(student_enc, H, NL).to(device)
    n_total = sum(p.numel() for p in model.parameters())
    n_enc = sum(p.numel() for p in model.encoder.parameters())
    n_distill_proj = sum(p.numel() for p in model.distill_projections.parameters())
    print(f"  Total: {n_total/1e6:.0f}M (encoder: {n_enc/1e6:.0f}M, "
          f"heads: {(n_total-n_enc-n_distill_proj)/1e6:.1f}M, "
          f"distill projections: {n_distill_proj/1e6:.1f}M)")

    student_out = f"{STUDENT_DIR}/{STUDENT_NAME.split('/')[-1]}-{suffix}"
    os.makedirs(student_out, exist_ok=True)

    # ─────────────────────────────────────────────────────
    # STAGE 1: Multi-loss distillation
    # ─────────────────────────────────────────────────────
    print(f"\n--- Stage 1: Distillation ({EPOCHS_S1} epochs) ---")
    shard_paths = sorted([f"{DISTILL_DIR}/{f}" for f in os.listdir(DISTILL_DIR)
                          if f.endswith(".parquet") and f.startswith("shard_")])
    dataset = DistillationDatasetV2(shard_paths, student_tok)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=collate, num_workers=2, pin_memory=True)

    enc_params = list(model.encoder.parameters()) + list(model.pred_embedding.parameters())
    head_params = [p for n, p in model.named_parameters()
                   if not n.startswith("encoder.") and not n.startswith("pred_embedding.")]
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENC},
        {"params": head_params, "lr": LR_HEADS}
    ], weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(EPOCHS_S1*len(loader)*S1_WARMUP), EPOCHS_S1*len(loader))
    T = 3.0; ALPHA = 0.7
    w = loss_weights

    for epoch in range(EPOCHS_S1):
        model.train()
        running = {"total": 0, "kl": 0, "hard": 0, "hidden": 0, "rdrop": 0, "n": 0}

        for batch in tqdm(loader, desc=f"S1 E{epoch+1}/{EPOCHS_S1}"):
            ids = batch["input_ids"].to(device); mask = batch["attention_mask"].to(device)
            pred = batch["predicate_idx"].to(device); valid = batch["valid"].to(device)
            B, S = ids.size(); fv = valid.view(B*S)

            # ── Forward pass 1 (with hidden states for distillation) ──
            out1 = model(ids, mask, pred, return_hidden_states=USE_HIDDEN_DISTILL)
            if USE_HIDDEN_DISTILL:
                s_pos, s_ner, s_arc, s_lab, s_srl, s_cls, student_hidden = out1
            else:
                s_pos, s_ner, s_arc, s_lab, s_srl, s_cls = out1

            # ── Logit KL (soft distillation) ──
            kl_pos = kl_loss(s_pos, batch["pos_t"].to(device), fv, T)
            kl_ner = kl_loss(s_ner, batch["ner_t"].to(device), fv, T)

            # SRL: filter by srl_valid
            sv = batch["srl_valid"].to(device)
            srl_mask = (valid & sv.unsqueeze(1)).view(B*S)
            kl_srl = kl_loss(s_srl, batch["srl_t"].to(device), srl_mask, T) if srl_mask.any() else torch.tensor(0.0, device=device)

            # CLS: per-sentence, scaled by avg tokens per sentence (compensate batch-size mismatch)
            tps = max(valid.sum().float().item()/B, 1.0)
            cls_ones = torch.ones(B, dtype=torch.bool, device=device)
            kl_cls = kl_loss(s_cls, batch["cls_t"].to(device), cls_ones, T) * tps

            # ── Hard CE ──
            hard_pos = hard_ce(s_pos, batch["pos_h"].to(device))
            hard_ner = hard_ce(s_ner, batch["ner_h"].to(device))
            hard_srl = hard_ce(s_srl, batch["srl_hard"].to(device)) if srl_mask.any() else torch.tensor(0.0, device=device)
            hard_cls = F.cross_entropy(s_cls, batch["cls_t"].to(device).argmax(-1))

            # ── DEP hard CE (head + relation) ──
            dh = batch["dep_head_hard"].to(device)
            dep_arc_loss = F.cross_entropy(s_arc.view(B*S, S), dh.view(B*S), ignore_index=-100)
            ph = dh.clamp(0).unsqueeze(-1).unsqueeze(-1).expand(B, S, 1, N_DEP)
            gl = s_lab.gather(2, ph).squeeze(2)
            dep_rel_loss = F.cross_entropy(gl.view(B*S, N_DEP),
                                           batch["dep_rel_hard"].to(device).view(B*S),
                                           ignore_index=-100)

            # ── Hidden state MSE (TinyBERT/PKD style) ──
            if USE_HIDDEN_DISTILL:
                hidden_loss = torch.tensor(0.0, device=device)
                for tl in TEACHER_LAYERS:
                    student_proj = model.distill_projections[f"l{tl}"](student_hidden[tl])
                    teacher_h = batch[f"teacher_hidden_l{tl}"].to(device)
                    hidden_loss = hidden_loss + hidden_state_mse(student_proj, teacher_h, fv)
                # Predicate-aware hidden state from sampled verb (last layer only)
                # The sampled verb's predicate-aware hidden is what student should produce
                # when its own predicate_idx is set to that verb's position
                # We use student's last layer for this (already predicate-aware via pred_embedding)
                if srl_mask.any():
                    # Use last layer of student (which is predicate-conditioned)
                    student_last = student_hidden[max(TEACHER_LAYERS)]
                    student_last_proj = model.distill_projections[f"l{max(TEACHER_LAYERS)}"](student_last)
                    pred_hidden_loss = hidden_state_mse(
                        student_last_proj, batch["teacher_pred_hidden"].to(device), srl_mask)
                    hidden_loss = hidden_loss + w["hidden_mult_pred"] * pred_hidden_loss
            else:
                hidden_loss = torch.tensor(0.0, device=device)

            # ── R-Drop (second forward pass with different dropout) ──
            if USE_RDROP:
                out2 = model(ids, mask, pred)
                _, s2_ner, _, _, s2_srl, s2_cls = out2
                rdrop_ner = rdrop_kl(s_ner, s2_ner, fv)
                rdrop_srl = rdrop_kl(s_srl, s2_srl, srl_mask) if srl_mask.any() else torch.tensor(0.0, device=device)
                rdrop_cls = rdrop_kl(s_cls.unsqueeze(1), s2_cls.unsqueeze(1),
                                     torch.ones(B, dtype=torch.bool, device=device))
                rdrop_total = rdrop_ner + rdrop_srl + rdrop_cls
            else:
                rdrop_total = torch.tensor(0.0, device=device)

            # ── Combine ──
            soft_total = (1.0*kl_pos + w["ner_mult"]*kl_ner +
                          w["srl_mult"]*kl_srl + w["cls_mult"]*kl_cls +
                          w["dep_arc"]*dep_arc_loss + w["dep_rel"]*dep_rel_loss)
            hard_total = (1.0*hard_pos + w["ner_mult"]*hard_ner +
                          w["srl_mult"]*hard_srl + w["cls_mult"]*hard_cls)

            loss = (ALPHA * soft_total
                    + (1-ALPHA) * w["alpha_hard"] * hard_total
                    + w["gamma_hidden"] * hidden_loss
                    + w["delta_rdrop"] * rdrop_total)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()

            running["total"] += loss.item()
            running["kl"] += soft_total.item()
            running["hard"] += hard_total.item()
            running["hidden"] += hidden_loss.item() if USE_HIDDEN_DISTILL else 0
            running["rdrop"] += rdrop_total.item() if USE_RDROP else 0
            running["n"] += 1

        n = running["n"]
        print(f"  S1 E{epoch+1} — Loss {running['total']/n:.3f}  "
              f"(kl {running['kl']/n:.3f}, hard {running['hard']/n:.3f}, "
              f"hidden {running['hidden']/n:.3f}, rdrop {running['rdrop']/n:.3f})")

    torch.save(model.state_dict(), f"{student_out}/model_s1.pt")
    print("Stage 1 done.\n")

    return model, student_tok, student_out

# Stage 2 will be added as a separate function; placeholder for now
print("train_student() defined.")
print("  Stage 1: distillation with logit KL + hard CE + hidden MSE + R-Drop")
print("  Stage 2 fine-tune will be a separate function (added in next cell)")

train_student() defined.
  Stage 1: distillation with logit KL + hard CE + hidden MSE + R-Drop
  Stage 2 fine-tune will be a separate function (added in next cell)


In [ ]:
# Cell 7: Stage 2 fine-tune — Pattern B (joint sequential SRL→CLS)
# Saves three checkpoints: best_srl, best_cls, best_combined

def extract_teacher_srl_silver(parquet_paths, max_per_shard=None):
    silver = []
    for path in parquet_paths:
        table = pq.read_table(path, columns=["words", "n_words", "per_verb_count",
                                              "per_verb_indices", "per_verb"])
        df = table.to_pandas()
        if max_per_shard:
            df = df.head(max_per_shard)
        for _, row in df.iterrows():
            words = list(row["words"])
            per_verb_data = json.loads(row["per_verb"])
            for pv in per_verb_data:
                silver.append({
                    "words": words,
                    "predicate_idx": pv["verb_idx"],
                    "srl_tags": pv["srl_hard"],
                })
    print(f"  Extracted {len(silver):,} per-verb SRL silver examples")
    return silver


def stage2_finetune(model, student_tok, student_out,
                    EPOCHS_S2=10, BATCH_SIZE=64,
                    LR_ENC=3e-6, LR_HEADS=1e-3,
                    SILVER_LIMIT=80000):
    """Stage 2 Pattern B: joint sequential SRL→CLS within each epoch.
    Saves: model.pt (best combined), model_best_srl.pt (best SRL alone), model_best_cls.pt (best CLS alone).
    """
    print(f"\n{'='*70}")
    print(f"--- Stage 2 Pattern B: SRL gold+silver + CLS ({EPOCHS_S2} epochs) ---")
    print(f"{'='*70}")

    for p in model.parameters(): p.requires_grad = True

    with open(f"{DATA}/srl_train.json") as f: srl_gold = json.load(f)
    with open(f"{DATA}/srl_dev.json") as f: srl_dev = json.load(f)

    print(f"\nLoading teacher silver SRL...")
    shard_paths = sorted([f"{DISTILL_DIR}/{f}" for f in os.listdir(DISTILL_DIR)
                          if f.endswith(".parquet") and f.startswith("shard_")])
    teacher_silver = extract_teacher_srl_silver(shard_paths)
    random.seed(42)
    if SILVER_LIMIT and len(teacher_silver) > SILVER_LIMIT:
        teacher_silver = random.sample(teacher_silver, SILVER_LIMIT)
    srl_combined = teacher_silver + srl_gold * 2
    random.shuffle(srl_combined)
    print(f"  SRL: {len(srl_combined):,} ({len(teacher_silver):,} silver + {len(srl_gold)*2:,} gold×2)")

    with open(f"{DATA}/cls_sgd_mwoz_train.json") as f: ca = json.load(f)
    random.seed(42); random.shuffle(ca); cls_dev_data = ca[:4000]; cls_train = ca[4000:]
    print(f"  CLS: {len(cls_train):,} train")

    class SRLDs(Dataset):
        def __init__(s, e, tok): s.e, s.tok = e, tok
        def __len__(s): return len(s.e)
        def __getitem__(s, i):
            ex = s.e[i]
            enc = s.tok(ex["words"], is_split_into_words=True, max_length=128,
                        padding="max_length", truncation=True, return_tensors="pt")
            pt, al, prev = 0, [], None
            for k, wid in enumerate(enc.word_ids()):
                if wid is None: al.append(-100)
                elif wid != prev:
                    al.append(srl_map.get(ex["srl_tags"][wid] if wid < len(ex["srl_tags"]) else "O", 0))
                    if wid == ex["predicate_idx"]: pt = k
                else: al.append(-100)
                prev = wid
            return {"input_ids": enc["input_ids"].squeeze(0),
                    "attention_mask": enc["attention_mask"].squeeze(0),
                    "labels": torch.tensor(al, dtype=torch.long),
                    "predicate_idx": torch.tensor(pt, dtype=torch.long)}

    class CLSDs(Dataset):
        def __init__(s, e, tok): s.e, s.tok = e, tok
        def __len__(s): return len(s.e)
        def __getitem__(s, i):
            ex = s.e[i]; t = ex.get("text",""); pt = ex.get("prev_text")
            if pt:
                enc = s.tok(pt, t, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
            else:
                enc = s.tok(t, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
            return {"input_ids": enc["input_ids"].squeeze(0),
                    "attention_mask": enc["attention_mask"].squeeze(0),
                    "labels": torch.tensor(cls_map[ex["cls_label"]], dtype=torch.long)}

    srl_ld = DataLoader(SRLDs(srl_combined, student_tok), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    cls_ld = DataLoader(CLSDs(cls_train, student_tok), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)

    enc_params = list(model.encoder.parameters()) + list(model.pred_embedding.parameters())
    head_params = [p for n, p in model.named_parameters()
                   if not n.startswith("encoder.") and not n.startswith("pred_embedding.")]
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENC},
        {"params": head_params, "lr": LR_HEADS}
    ], weight_decay=0.01)
    total_steps = EPOCHS_S2 * (len(srl_ld) + len(cls_ld))
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps*0.1), total_steps)

    best_combined = 0.0; best_srl = 0.0; best_cls = 0.0
    history = []

    for epoch in range(EPOCHS_S2):
        model.train()
        for b in tqdm(srl_ld, desc=f"S2-SRL E{epoch+1}", leave=False):
            o = model(b["input_ids"].to(device), b["attention_mask"].to(device),
                      b["predicate_idx"].to(device))[4]
            loss = F.cross_entropy(o.view(-1, N_SRL), b["labels"].to(device).view(-1), ignore_index=-100)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
        for b in tqdm(cls_ld, desc=f"S2-CLS E{epoch+1}", leave=False):
            o = model(b["input_ids"].to(device), b["attention_mask"].to(device),
                      torch.zeros(b["input_ids"].size(0), dtype=torch.long, device=device))[5]
            loss = F.cross_entropy(o, b["labels"].to(device))
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()

        # Eval
        model.eval()
        sc_arg, sc_vit, st = 0, 0, 0
        with torch.no_grad():
            for ex in srl_dev[:300]:
                enc = student_tok(ex["words"], is_split_into_words=True, return_tensors="pt",
                                  padding=True, truncation=True, max_length=128)
                pt = 0; prev = None
                for k, wid in enumerate(enc.word_ids()):
                    if wid is not None and wid != prev and wid == ex["predicate_idx"]: pt = k; break
                    prev = wid
                o = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                          torch.tensor([pt], dtype=torch.long, device=device))[4]
                vi = []; prev = None
                for k, wid in enumerate(enc.word_ids()):
                    if wid is not None and wid != prev and wid < len(ex["srl_tags"]): vi.append((k, wid))
                    prev = wid
                if not vi: continue
                valid_logits = o[0, [k for k, _ in vi]].cpu()
                vit_path = viterbi_decode_srl(valid_logits)
                for idx, (k, wid) in enumerate(vi):
                    gold = ex["srl_tags"][wid]
                    if SRL_TAGS[o[0, k].argmax().item()] == gold: sc_arg += 1
                    if SRL_TAGS[vit_path[idx]] == gold: sc_vit += 1
                    st += 1
        srl_arg = sc_arg/st if st > 0 else 0
        srl_vit = sc_vit/st if st > 0 else 0

        cc, ct = 0, 0
        with torch.no_grad():
            for ex in cls_dev_data[:500]:
                text = ex.get("text", ""); pt = ex.get("prev_text")
                if pt: enc = student_tok(pt, text, return_tensors="pt", padding=True, truncation=True, max_length=128)
                else: enc = student_tok(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
                o = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                          torch.zeros(1, dtype=torch.long, device=device))[5]
                if CLS_LABELS[o.argmax(-1).item()] == ex["cls_label"]: cc += 1
                ct += 1
        cls_a = cc/ct if ct > 0 else 0

        history.append({"epoch": epoch+1, "srl_arg": srl_arg, "srl_vit": srl_vit, "cls": cls_a})

        combined = srl_vit + cls_a
        markers = []

        # Save best combined
        if combined > best_combined:
            best_combined = combined
            torch.save(model.state_dict(), f"{student_out}/model.pt")
            student_tok.save_pretrained(student_out)
            markers.append(f"BEST COMBINED ({combined:.4f})")

        # Save best SRL alone (won't be hurt by mid-run CLS regressions)
        if srl_vit > best_srl:
            best_srl = srl_vit
            torch.save(model.state_dict(), f"{student_out}/model_best_srl.pt")
            markers.append(f"BEST SRL ({srl_vit:.4f})")

        # Save best CLS alone
        if cls_a > best_cls:
            best_cls = cls_a
            torch.save(model.state_dict(), f"{student_out}/model_best_cls.pt")
            markers.append(f"BEST CLS ({cls_a:.4f})")

        marker_str = "  ★ " + ", ".join(markers) if markers else ""
        print(f"  S2 E{epoch+1} — SRL argmax: {srl_arg:.4f}, Viterbi: {srl_vit:.4f}, CLS: {cls_a:.4f}{marker_str}")

    # Save metadata
    meta_path = f"{student_out}/training_log.json"
    with open(meta_path, "w") as f:
        json.dump({
            "stage": "Stage 2 Pattern B",
            "best_combined": best_combined, "best_srl": best_srl, "best_cls": best_cls,
            "epochs": EPOCHS_S2, "batch_size": BATCH_SIZE,
            "lr_enc": LR_ENC, "lr_heads": LR_HEADS,
            "silver_limit": SILVER_LIMIT,
            "history": history,
        }, f, indent=2)
    print(f"\n{'='*70}")
    print(f"Pattern B done: best_srl={best_srl:.4f}, best_cls={best_cls:.4f}, "
          f"best_combined={best_combined:.4f}")
    print(f"  3 checkpoints saved: model.pt (combined), model_best_srl.pt, model_best_cls.pt")
    print(f"{'='*70}")
    return best_combined


print("stage2_finetune (Pattern B) updated:")
print("  Saves 3 checkpoints: best combined, best SRL alone, best CLS alone")
print("  Saves training_log.json with full epoch history")

stage2_finetune (Pattern B) updated:
  Saves 3 checkpoints: best combined, best SRL alone, best CLS alone
  Saves training_log.json with full epoch history


In [ ]:
# Cell 8: Run xsmall first — fastest, smallest, biggest gap to close

model, tok, out_dir = train_student(
    "microsoft/deberta-v3-xsmall",
    suffix="v2",
    EPOCHS_S1=8,
    BATCH_SIZE=64,
    USE_RDROP=True,
    USE_HIDDEN_DISTILL=True,
)

best = stage2_finetune(model, tok, out_dir, EPOCHS_S2=10, BATCH_SIZE=64)
print(f"\n✓ xsmall v2 done. Best combined SRL+CLS: {best:.4f}")

TRAINING microsoft/deberta-v3-xsmall v2 (v2)
  Stage 1: 8 epochs distillation
  Stage 2a: 8 epochs SRL gold+silver fine-tune
  R-Drop: True, Hidden distillation: True


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/241M [00:00<?, ?B/s]

...

  S2 E6 — SRL argmax: 0.8576, Viterbi: 0.8567, CLS: 0.9460


  S2 E7 — SRL argmax: 0.8592, Viterbi: 0.8616, CLS: 0.9380


  S2 E8 — SRL argmax: 0.8697, Viterbi: 0.8754, CLS: 0.9420
    Saved (SRL Vit: 0.8754, CLS: 0.9420)


  S2 E9 — SRL argmax: 0.8680, Viterbi: 0.8714, CLS: 0.9440


  S2 E10 — SRL argmax: 0.8623, Viterbi: 0.8628, CLS: 0.9400

Stage 2 complete — Best combined: 1.8174

✓ xsmall v2 done. Best combined SRL+CLS: 1.8174


In [ ]:
# Cell 9: Stage 2 Pattern C (sequential SRL-only → CLS-only frozen encoder)
# Saves: model.pt (final), model_s2a_best.pt (best SRL), training_log.json

def stage2_finetune_pattern_c(student_name, suffix_in="v2", suffix_out="v2-patC",
                              EPOCHS_S2A=8, EPOCHS_S2B=4, BATCH_SIZE=64,
                              LR_ENC=3e-6, LR_HEADS=1e-3, LR_CLS_S2B=5e-4,
                              SILVER_LIMIT=80000):
    """Pattern C: 2-stage Stage 2.
    Stage 2a: SRL-only fine-tune (encoder + SRL components trainable, CLS frozen)
    Stage 2b: CLS-only fine-tune (encoder + SRL frozen, only CLS components train)
    """
    print(f"\n{'='*70}")
    print(f"PATTERN C: Sequential SRL-only → CLS-only (frozen encoder)")
    print(f"  Loading Stage 1 from: {student_name}-{suffix_in}/model_s1.pt")
    print(f"{'='*70}")

    student_tok = AutoTokenizer.from_pretrained(student_name)
    student_enc = AutoModel.from_pretrained(student_name).float()
    H = student_enc.config.hidden_size
    NL = student_enc.config.num_hidden_layers + 1
    model = StudentCascadeV5(student_enc, H, NL).to(device)

    s1_path = f"{STUDENT_DIR}/{student_name.split('/')[-1]}-{suffix_in}/model_s1.pt"
    model.load_state_dict(torch.load(s1_path, map_location="cpu"))
    print(f"  ✓ Loaded Stage 1 checkpoint")

    student_out = f"{STUDENT_DIR}/{student_name.split('/')[-1]}-{suffix_out}"
    os.makedirs(student_out, exist_ok=True)

    with open(f"{DATA}/srl_train.json") as f: srl_gold = json.load(f)
    with open(f"{DATA}/srl_dev.json") as f: srl_dev = json.load(f)
    print(f"\nLoading teacher silver SRL...")
    shard_paths = sorted([f"{DISTILL_DIR}/{f}" for f in os.listdir(DISTILL_DIR)
                          if f.endswith(".parquet") and f.startswith("shard_")])
    teacher_silver = extract_teacher_srl_silver(shard_paths)
    random.seed(42)
    if SILVER_LIMIT and len(teacher_silver) > SILVER_LIMIT:
        teacher_silver = random.sample(teacher_silver, SILVER_LIMIT)
    srl_combined = teacher_silver + srl_gold * 2
    random.shuffle(srl_combined)
    print(f"  SRL: {len(srl_combined):,}")

    with open(f"{DATA}/cls_sgd_mwoz_train.json") as f: ca = json.load(f)
    random.seed(42); random.shuffle(ca); cls_dev_data = ca[:4000]; cls_train = ca[4000:]
    print(f"  CLS: {len(cls_train):,} train")

    class SRLDs(Dataset):
        def __init__(s, e, tok): s.e, s.tok = e, tok
        def __len__(s): return len(s.e)
        def __getitem__(s, i):
            ex = s.e[i]
            enc = s.tok(ex["words"], is_split_into_words=True, max_length=128,
                        padding="max_length", truncation=True, return_tensors="pt")
            pt, al, prev = 0, [], None
            for k, wid in enumerate(enc.word_ids()):
                if wid is None: al.append(-100)
                elif wid != prev:
                    al.append(srl_map.get(ex["srl_tags"][wid] if wid < len(ex["srl_tags"]) else "O", 0))
                    if wid == ex["predicate_idx"]: pt = k
                else: al.append(-100)
                prev = wid
            return {"input_ids": enc["input_ids"].squeeze(0),
                    "attention_mask": enc["attention_mask"].squeeze(0),
                    "labels": torch.tensor(al, dtype=torch.long),
                    "predicate_idx": torch.tensor(pt, dtype=torch.long)}

    class CLSDs(Dataset):
        def __init__(s, e, tok): s.e, s.tok = e, tok
        def __len__(s): return len(s.e)
        def __getitem__(s, i):
            ex = s.e[i]; t = ex.get("text",""); pt = ex.get("prev_text")
            if pt:
                enc = s.tok(pt, t, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
            else:
                enc = s.tok(t, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
            return {"input_ids": enc["input_ids"].squeeze(0),
                    "attention_mask": enc["attention_mask"].squeeze(0),
                    "labels": torch.tensor(cls_map[ex["cls_label"]], dtype=torch.long)}

    srl_ld = DataLoader(SRLDs(srl_combined, student_tok), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    cls_ld = DataLoader(CLSDs(cls_train, student_tok), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)

    history_s2a = []; history_s2b = []

    # ── Stage 2a: SRL-only ──
    print(f"\n--- Stage 2a: SRL-only ({EPOCHS_S2A} epochs) ---")
    for p in model.parameters(): p.requires_grad = False
    for p in model.encoder.parameters(): p.requires_grad = True
    for p in model.pred_embedding.parameters(): p.requires_grad = True
    srl_modules = [model.srl_sm, model.srl_interaction_proj, model.srl_lstm,
                   model.srl_proj, model.srl_head,
                   model.dep_sm, model.dep_proj, model.dep_lstm, model.dep_lstm_proj, model.dep_biaff,
                   model.pos_sm, model.pos_head,
                   model.ner_sm, model.ner_lstm, model.ner_proj, model.ner_head]
    for m in srl_modules:
        for p in m.parameters(): p.requires_grad = True
    print(f"  Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M")

    enc_params = list(model.encoder.parameters()) + list(model.pred_embedding.parameters())
    head_params = [p for n, p in model.named_parameters()
                   if p.requires_grad and not n.startswith("encoder.") and not n.startswith("pred_embedding.")]
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENC},
        {"params": head_params, "lr": LR_HEADS}
    ], weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(EPOCHS_S2A*len(srl_ld)*0.1), EPOCHS_S2A*len(srl_ld))
    best_srl = 0.0

    for epoch in range(EPOCHS_S2A):
        model.train()
        for b in tqdm(srl_ld, desc=f"S2a-SRL E{epoch+1}", leave=False):
            o = model(b["input_ids"].to(device), b["attention_mask"].to(device),
                      b["predicate_idx"].to(device))[4]
            loss = F.cross_entropy(o.view(-1, N_SRL), b["labels"].to(device).view(-1), ignore_index=-100)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()

        model.eval(); sc_arg, sc_vit, st = 0, 0, 0
        with torch.no_grad():
            for ex in srl_dev[:300]:
                enc = student_tok(ex["words"], is_split_into_words=True, return_tensors="pt",
                                  padding=True, truncation=True, max_length=128)
                pt = 0; prev = None
                for k, wid in enumerate(enc.word_ids()):
                    if wid is not None and wid != prev and wid == ex["predicate_idx"]: pt = k; break
                    prev = wid
                o = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                          torch.tensor([pt], dtype=torch.long, device=device))[4]
                vi = []; prev = None
                for k, wid in enumerate(enc.word_ids()):
                    if wid is not None and wid != prev and wid < len(ex["srl_tags"]): vi.append((k, wid))
                    prev = wid
                if not vi: continue
                valid_logits = o[0, [k for k, _ in vi]].cpu()
                vit_path = viterbi_decode_srl(valid_logits)
                for idx, (k, wid) in enumerate(vi):
                    gold = ex["srl_tags"][wid]
                    if SRL_TAGS[o[0, k].argmax().item()] == gold: sc_arg += 1
                    if SRL_TAGS[vit_path[idx]] == gold: sc_vit += 1
                    st += 1
        srl_arg = sc_arg/st if st > 0 else 0
        srl_vit = sc_vit/st if st > 0 else 0
        history_s2a.append({"epoch": epoch+1, "srl_arg": srl_arg, "srl_vit": srl_vit})
        marker = ""
        if srl_vit > best_srl:
            best_srl = srl_vit
            torch.save(model.state_dict(), f"{student_out}/model_s2a_best.pt")
            marker = f"  ★ BEST SRL ({srl_vit:.4f})"
        print(f"  S2a E{epoch+1} — SRL argmax: {srl_arg:.4f}, Viterbi: {srl_vit:.4f}{marker}")

    print(f"\n  Loading best SRL checkpoint (Vit {best_srl:.4f}) for Stage 2b...")
    model.load_state_dict(torch.load(f"{student_out}/model_s2a_best.pt", map_location="cpu"))

    # ── Stage 2b: CLS-only ──
    print(f"\n--- Stage 2b: CLS-only, encoder + SRL frozen ({EPOCHS_S2B} epochs) ---")
    for p in model.parameters(): p.requires_grad = False
    for m in [model.cls_sm, model.cls_pool, model.cls_head]:
        for p in m.parameters(): p.requires_grad = True
    print(f"  Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M (CLS only)")

    cls_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(cls_params, lr=LR_CLS_S2B, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(EPOCHS_S2B*len(cls_ld)*0.1), EPOCHS_S2B*len(cls_ld))
    best_cls = 0.0

    for epoch in range(EPOCHS_S2B):
        model.train()
        model.encoder.eval()
        for layer_attr in ["pos_sm", "ner_lstm", "ner_proj", "ner_head",
                           "dep_proj", "dep_lstm", "dep_biaff",
                           "srl_sm", "srl_interaction_proj", "srl_lstm", "srl_proj", "srl_head"]:
            getattr(model, layer_attr).eval()

        for b in tqdm(cls_ld, desc=f"S2b-CLS E{epoch+1}", leave=False):
            o = model(b["input_ids"].to(device), b["attention_mask"].to(device),
                      torch.zeros(b["input_ids"].size(0), dtype=torch.long, device=device))[5]
            loss = F.cross_entropy(o, b["labels"].to(device))
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()

        model.eval()
        sc_vit, st = 0, 0
        with torch.no_grad():
            for ex in srl_dev[:300]:
                enc = student_tok(ex["words"], is_split_into_words=True, return_tensors="pt",
                                  padding=True, truncation=True, max_length=128)
                pt = 0; prev = None
                for k, wid in enumerate(enc.word_ids()):
                    if wid is not None and wid != prev and wid == ex["predicate_idx"]: pt = k; break
                    prev = wid
                o = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                          torch.tensor([pt], dtype=torch.long, device=device))[4]
                vi = []; prev = None
                for k, wid in enumerate(enc.word_ids()):
                    if wid is not None and wid != prev and wid < len(ex["srl_tags"]): vi.append((k, wid))
                    prev = wid
                if not vi: continue
                valid_logits = o[0, [k for k, _ in vi]].cpu()
                vit_path = viterbi_decode_srl(valid_logits)
                for idx, (k, wid) in enumerate(vi):
                    gold = ex["srl_tags"][wid]
                    if SRL_TAGS[vit_path[idx]] == gold: sc_vit += 1
                    st += 1
        srl_vit = sc_vit/st if st > 0 else 0

        cc, ct = 0, 0
        with torch.no_grad():
            for ex in cls_dev_data[:500]:
                text = ex.get("text", ""); pt = ex.get("prev_text")
                if pt: enc = student_tok(pt, text, return_tensors="pt", padding=True, truncation=True, max_length=128)
                else: enc = student_tok(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
                o = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                          torch.zeros(1, dtype=torch.long, device=device))[5]
                if CLS_LABELS[o.argmax(-1).item()] == ex["cls_label"]: cc += 1
                ct += 1
        cls_a = cc/ct if ct > 0 else 0
        history_s2b.append({"epoch": epoch+1, "srl_vit_frozen": srl_vit, "cls": cls_a})

        marker = ""
        if cls_a > best_cls:
            best_cls = cls_a
            torch.save(model.state_dict(), f"{student_out}/model.pt")
            student_tok.save_pretrained(student_out)
            marker = f"  ★ BEST CLS ({cls_a:.4f})"
        print(f"  S2b E{epoch+1} — SRL Vit (frozen): {srl_vit:.4f}, CLS: {cls_a:.4f}{marker}")

    # Save metadata
    with open(f"{student_out}/training_log.json", "w") as f:
        json.dump({
            "stage": "Stage 2 Pattern C",
            "best_srl": best_srl, "best_cls": best_cls,
            "epochs_s2a": EPOCHS_S2A, "epochs_s2b": EPOCHS_S2B,
            "batch_size": BATCH_SIZE,
            "lr_enc": LR_ENC, "lr_heads": LR_HEADS, "lr_cls_s2b": LR_CLS_S2B,
            "silver_limit": SILVER_LIMIT,
            "history_s2a": history_s2a, "history_s2b": history_s2b,
        }, f, indent=2)

    print(f"\n{'='*70}")
    print(f"PATTERN C COMPLETE — Best SRL: {best_srl:.4f}, Best CLS: {best_cls:.4f}")
    print(f"  Saved: model.pt (final), model_s2a_best.pt (best SRL), training_log.json")
    print(f"  → {student_out}")
    print(f"{'='*70}")
    return best_srl, best_cls


print("stage2_finetune_pattern_c updated:")
print("  Saves training_log.json with full history (S2a + S2b)")

stage2_finetune_pattern_c updated:
  Saves training_log.json with full history (S2a + S2b)


In [ ]:
# Cell 10: Compare Pattern B vs Pattern C — full 5-head eval + SRL error analysis

from collections import Counter, defaultdict


def evaluate_5_heads(model, tok, ud_dev, ner_dev, srl_dev, cls_dev_data, n_eval=500):
    """Run full 5-head eval. Returns dict of head accuracies + Viterbi for SRL."""
    model.eval()
    results = {}
    for task, data in [("POS", ud_dev), ("NER", ner_dev), ("DEP", ud_dev),
                        ("SRL", srl_dev), ("CLS", cls_dev_data)]:
        c, t, c_vit = 0, 0, 0
        with torch.no_grad():
            for ex in tqdm(data[:n_eval], desc=task, leave=False):
                if task in ("POS", "NER", "DEP"):
                    enc = tok(ex["words"], is_split_into_words=True, return_tensors="pt",
                              padding=True, truncation=True, max_length=128)
                    out = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                                torch.zeros(1, dtype=torch.long, device=device))
                    w2t = {}; prev = None
                    for k, wid in enumerate(enc.word_ids()):
                        if wid is not None and wid != prev: w2t[wid] = k
                        prev = wid
                    if task == "POS":
                        for wid, tidx in w2t.items():
                            if wid < len(ex["pos_tags"]):
                                if POS_LABELS[out[0][0, tidx].argmax().item()] == ex["pos_tags"][wid]: c += 1
                                t += 1
                    elif task == "NER":
                        for wid, tidx in w2t.items():
                            if wid < len(ex.get("ner_tags", [])):
                                if NER_LABELS[out[1][0, tidx].argmax().item()] == ex["ner_tags"][wid]: c += 1
                                t += 1
                    elif task == "DEP":
                        t2w = {v: k for k, v in w2t.items()}
                        for wid, tidx in w2t.items():
                            if wid >= len(ex["heads"]): continue
                            gold = ex["heads"][wid]
                            pred_wid = t2w.get(out[2][0, tidx].argmax().item(), -99)
                            if gold == -1:
                                if pred_wid == wid: c += 1
                            elif pred_wid == gold: c += 1
                            t += 1
                elif task == "SRL":
                    enc = tok(ex["words"], is_split_into_words=True, return_tensors="pt",
                              padding=True, truncation=True, max_length=128)
                    pt = 0; prev = None
                    for k, wid in enumerate(enc.word_ids()):
                        if wid is not None and wid != prev and wid == ex["predicate_idx"]: pt = k; break
                        prev = wid
                    out = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                                torch.tensor([pt], dtype=torch.long, device=device))
                    vi = []; prev = None
                    for k, wid in enumerate(enc.word_ids()):
                        if wid is not None and wid != prev and wid < len(ex["srl_tags"]): vi.append((k, wid))
                        prev = wid
                    if not vi: continue
                    valid_logits = out[4][0, [k for k, _ in vi]].cpu()
                    vit_path = viterbi_decode_srl(valid_logits)
                    for idx, (k, wid) in enumerate(vi):
                        gold = ex["srl_tags"][wid]
                        if SRL_TAGS[out[4][0, k].argmax().item()] == gold: c += 1
                        if SRL_TAGS[vit_path[idx]] == gold: c_vit += 1
                        t += 1
                elif task == "CLS":
                    text = ex.get("text", ""); pt = ex.get("prev_text")
                    if pt: enc = tok(pt, text, return_tensors="pt", padding=True, truncation=True, max_length=128)
                    else: enc = tok(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
                    out = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                                torch.zeros(1, dtype=torch.long, device=device))
                    if CLS_LABELS[out[5].argmax(-1).item()] == ex["cls_label"]: c += 1
                    t += 1
        results[task] = c / t if t > 0 else 0
        if task == "SRL":
            results["SRL_vit"] = c_vit / t if t > 0 else 0
    return results


def srl_error_analysis(model, tok, srl_dev, n_eval=500):
    """Detailed SRL error analysis. Returns dict with error categories, per-tag accuracy,
    confusions, span F1, hardest verbs, BIO transition validity.
    """
    model.eval()
    cats = {"miss": 0, "halluc": 0, "boundary": 0, "role": 0, "total_errors": 0, "total_tokens": 0}
    confusion_vit = Counter()
    error_by_type = defaultdict(lambda: {"correct_arg": 0, "correct_vit": 0, "total": 0})
    sent_lengths = defaultdict(lambda: {"correct_arg": 0, "correct_vit": 0, "total": 0})
    verb_stats = defaultdict(lambda: {"correct": 0, "total": 0})
    span_g, span_p, span_m = 0, 0, 0
    invalid_argmax, invalid_viterbi, total_transitions = 0, 0, 0

    def cat_err(g, p):
        if g == "O" and p != "O": return "halluc"
        if g != "O" and p == "O": return "miss"
        if g != "O" and p != "O":
            gp, gr = (g[0], g[2:]) if "-" in g else (g, g)
            pp, pr = (p[0], p[2:]) if "-" in p else (p, p)
            if gr == pr and gp != pp: return "boundary"
            return "role"
        return None

    def extract_spans(seq):
        spans = []; i = 0
        while i < len(seq):
            if seq[i].startswith("B-"):
                role = seq[i][2:]; start = i; i += 1
                while i < len(seq) and seq[i] == f"I-{role}": i += 1
                spans.append((role, start, i - 1))
            elif seq[i] == "V":
                spans.append(("V", i, i)); i += 1
            else: i += 1
        return spans

    with torch.no_grad():
        for ex in tqdm(srl_dev[:n_eval], desc="error analysis", leave=False):
            words = ex["words"]; pred_word_idx = ex["predicate_idx"]; n = len(words)
            enc = tok(words, is_split_into_words=True, return_tensors="pt",
                      padding=True, truncation=True, max_length=128)
            pt = 0; prev = None
            for k, wid in enumerate(enc.word_ids()):
                if wid is not None and wid != prev and wid == pred_word_idx: pt = k; break
                prev = wid
            srl_logits = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                               torch.tensor([pt], dtype=torch.long, device=device))[4]
            vi = []; prev = None
            for k, wid in enumerate(enc.word_ids()):
                if wid is not None and wid != prev and wid < len(ex["srl_tags"]): vi.append((k, wid))
                prev = wid
            if not vi: continue
            valid_logits = srl_logits[0, [k for k, _ in vi]].cpu()
            vit_path = viterbi_decode_srl(valid_logits)

            gold_seq, pred_arg_seq, pred_vit_seq = [], [], []
            for idx, (k, wid) in enumerate(vi):
                g = ex["srl_tags"][wid]
                p_arg = SRL_TAGS[srl_logits[0, k].argmax().item()]
                p_vit = SRL_TAGS[vit_path[idx]]
                gold_seq.append(g); pred_arg_seq.append(p_arg); pred_vit_seq.append(p_vit)

                lb = "short" if n <= 10 else "medium" if n <= 25 else "long"
                error_by_type[g]["total"] += 1
                sent_lengths[lb]["total"] += 1
                if g == p_arg: error_by_type[g]["correct_arg"] += 1; sent_lengths[lb]["correct_arg"] += 1
                if g == p_vit: error_by_type[g]["correct_vit"] += 1; sent_lengths[lb]["correct_vit"] += 1

                cats["total_tokens"] += 1
                if g != p_vit:
                    cats["total_errors"] += 1
                    cat = cat_err(g, p_vit)
                    if cat: cats[cat] += 1
                    confusion_vit[(g, p_vit)] += 1

                verb = words[pred_word_idx] if pred_word_idx < len(words) else "?"
                verb_stats[verb]["total"] += 1
                if g == p_vit: verb_stats[verb]["correct"] += 1

            # Span F1 (Viterbi)
            gs = set(extract_spans(gold_seq)); ps = set(extract_spans(pred_vit_seq))
            span_g += len(gs); span_p += len(ps); span_m += len(gs & ps)
            # Invalid BIO transitions
            for seq, key in [(pred_arg_seq, "argmax"), (pred_vit_seq, "viterbi")]:
                for i in range(1, len(seq)):
                    total_transitions += (1 if seq is pred_vit_seq else 0)
                    if seq[i].startswith("I-"):
                        role = seq[i][2:]
                        if seq[i-1] != f"B-{role}" and seq[i-1] != f"I-{role}":
                            if key == "argmax": invalid_argmax += 1
                            else: invalid_viterbi += 1
            total_transitions += len(pred_arg_seq) - 1  # double-count fix

    # Span-level F1
    prec = span_m / span_p if span_p > 0 else 0
    rec = span_m / span_g if span_g > 0 else 0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0

    return {
        "cats": cats,
        "confusion_top": dict(confusion_vit.most_common(20)),
        "per_tag": dict(error_by_type),
        "sent_lengths": dict(sent_lengths),
        "verb_stats": dict(verb_stats),
        "span_f1": (prec, rec, f1),
        "invalid_transitions": {"argmax": invalid_argmax, "viterbi": invalid_viterbi,
                                 "total": total_transitions // 2},
    }


# ── Load eval data ──
print("Loading eval data...")
with open(f"{DATA}/ud_dev.json") as f: ud_dev = json.load(f)
with open(f"{DATA}/ner_spanmarker_dev.json") as f: ner_dev = json.load(f)
with open(f"{DATA}/srl_dev.json") as f: srl_dev = json.load(f)
with open(f"{DATA}/cls_sgd_mwoz_train.json") as f: cls_data = json.load(f)
random.seed(42); random.shuffle(cls_data); cls_dev_data = cls_data[:4000]


# ── Helper: load + eval a model variant ──
def load_and_eval(student_name, suffix):
    path = f"{STUDENT_DIR}/{student_name.split('/')[-1]}-{suffix}/model.pt"
    print(f"\n{'='*70}\nLOADING + EVALUATING: {suffix}\n{'='*70}")
    print(f"  {path}")
    tok = AutoTokenizer.from_pretrained(student_name)
    enc = AutoModel.from_pretrained(student_name).float()
    H = enc.config.hidden_size; NL = enc.config.num_hidden_layers + 1
    m = StudentCascadeV5(enc, H, NL).to(device)
    m.load_state_dict(torch.load(path, map_location="cpu"))
    results = evaluate_5_heads(m, tok, ud_dev, ner_dev, srl_dev, cls_dev_data)
    err = srl_error_analysis(m, tok, srl_dev)
    del m, enc; torch.cuda.empty_cache()
    return results, err


STUDENT_NAME = "microsoft/deberta-v3-xsmall"
patB_results, patB_err = load_and_eval(STUDENT_NAME, "v2")
patC_results, patC_err = load_and_eval(STUDENT_NAME, "v2-patC")


# ─────────────────────────────────────────────────────
# COMPARISON TABLES
# ─────────────────────────────────────────────────────
v5_baseline = {"POS": 0.971, "NER": 0.983, "DEP": 0.942, "SRL": 0.875, "SRL_vit": 0.877, "CLS": 0.942}
teacher = {"POS": 0.977, "NER": 0.978, "DEP": 0.944, "SRL": 0.920, "SRL_vit": 0.920, "CLS": 0.951}

print(f"\n\n{'='*80}")
print(f"FULL 5-HEAD COMPARISON")
print(f"{'='*80}")
print(f"  {'Head':<10} {'v5 base':>9}  {'patB':>9}  {'patC':>9}  {'Teacher':>9}  {'B vs v5':>9}  {'C vs v5':>9}")
print(f"  {'─'*10} {'─'*9}  {'─'*9}  {'─'*9}  {'─'*9}  {'─'*9}  {'─'*9}")
for h in ["POS", "NER", "DEP", "SRL", "SRL_vit", "CLS"]:
    v5 = v5_baseline[h]; b = patB_results[h]; c = patC_results[h]; te = teacher[h]
    print(f"  {h:<10} {v5:>9.4f}  {b:>9.4f}  {c:>9.4f}  {te:>9.4f}  "
          f"{b-v5:>+9.4f}  {c-v5:>+9.4f}")
print(f"{'='*80}")


# ─────────────────────────────────────────────────────
# SRL ERROR BREAKDOWN — side by side
# ─────────────────────────────────────────────────────
v5_cats = {"miss": 353, "halluc": 194, "boundary": 54, "role": 456, "total_errors": 1057}
print(f"\n{'='*80}")
print(f"SRL ERROR CATEGORIES (Viterbi)")
print(f"{'='*80}")
print(f"  {'Category':<35} {'v5':>8} {'patB':>8} {'patC':>8} {'B-v5':>8} {'C-v5':>8}")
print(f"  {'─'*35} {'─'*8} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
for cat in ["miss", "halluc", "boundary", "role", "total_errors"]:
    label = {"miss": "Missed args (gold≠O, pred=O)",
             "halluc": "Hallucinated (gold=O, pred≠O)",
             "boundary": "Boundary errors (B↔I)",
             "role": "Role confusion",
             "total_errors": "TOTAL ERRORS"}[cat]
    v5_v = v5_cats[cat]; b_v = patB_err["cats"][cat]; c_v = patC_err["cats"][cat]
    print(f"  {label:<35} {v5_v:>8} {b_v:>8} {c_v:>8} {b_v-v5_v:>+8} {c_v-v5_v:>+8}")


# ─────────────────────────────────────────────────────
# SPAN-LEVEL F1
# ─────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"SPAN-LEVEL F1 (Viterbi)")
print(f"{'='*80}")
for label, err in [("patB", patB_err), ("patC", patC_err)]:
    p, r, f = err["span_f1"]
    print(f"  {label:<8} P={p:.4f} R={r:.4f} F1={f:.4f}")
print(f"  {'v5 (ref)':<8} P=0.840 R=0.869 F1=0.854")


# ─────────────────────────────────────────────────────
# PER-TAG ACCURACY (only common tags) — side by side
# ─────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"PER-TAG ACCURACY (Viterbi, tags with ≥20 occurrences)")
print(f"{'='*80}")
print(f"  {'Tag':<15} {'patB':>8} {'patC':>8} {'C-B':>8}  {'Count':>7}")
print(f"  {'─'*15} {'─'*8} {'─'*8} {'─'*8}  {'─'*7}")
all_tags = set(patB_err["per_tag"].keys()) | set(patC_err["per_tag"].keys())
sorted_tags = sorted(all_tags, key=lambda t: -patB_err["per_tag"].get(t, {"total":0})["total"])
for tag in sorted_tags:
    info_b = patB_err["per_tag"].get(tag, {"correct_vit": 0, "total": 0})
    info_c = patC_err["per_tag"].get(tag, {"correct_vit": 0, "total": 0})
    if max(info_b["total"], info_c["total"]) >= 20:
        acc_b = info_b["correct_vit"] / max(info_b["total"], 1)
        acc_c = info_c["correct_vit"] / max(info_c["total"], 1)
        print(f"  {tag:<15} {acc_b:>8.3f} {acc_c:>8.3f} {acc_c-acc_b:>+8.3f}  {info_b['total']:>7}")


# ─────────────────────────────────────────────────────
# TOP CONFUSIONS — side by side
# ─────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"TOP 10 CONFUSIONS (Viterbi, gold → pred)")
print(f"{'='*80}")
print(f"  {'patB':<45}  {'patC':<45}")
print(f"  {'─'*45}  {'─'*45}")
patB_conf = list(patB_err["confusion_top"].items())[:10]
patC_conf = list(patC_err["confusion_top"].items())[:10]
for i in range(max(len(patB_conf), len(patC_conf))):
    b_str = ""; c_str = ""
    if i < len(patB_conf):
        (g, p), n = patB_conf[i]; b_str = f"{g:>15} → {p:<15} {n:5d}"
    if i < len(patC_conf):
        (g, p), n = patC_conf[i]; c_str = f"{g:>15} → {p:<15} {n:5d}"
    print(f"  {b_str:<45}  {c_str:<45}")


# ─────────────────────────────────────────────────────
# ACCURACY BY SENTENCE LENGTH
# ─────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"ACCURACY BY SENTENCE LENGTH (Viterbi)")
print(f"{'='*80}")
print(f"  {'Bucket':<12} {'patB':>8} {'patC':>8} {'C-B':>8}  {'#tokens':>8}")
print(f"  {'─'*12} {'─'*8} {'─'*8} {'─'*8}  {'─'*8}")
for bucket in ["short", "medium", "long"]:
    info_b = patB_err["sent_lengths"].get(bucket, {"correct_vit":0, "total":1})
    info_c = patC_err["sent_lengths"].get(bucket, {"correct_vit":0, "total":1})
    acc_b = info_b["correct_vit"] / max(info_b["total"], 1)
    acc_c = info_c["correct_vit"] / max(info_c["total"], 1)
    label = f"{bucket} ({'≤10' if bucket=='short' else '≤25' if bucket=='medium' else '>25'})"
    print(f"  {label:<12} {acc_b:>8.4f} {acc_c:>8.4f} {acc_c-acc_b:>+8.4f}  {info_b['total']:>8}")


# ─────────────────────────────────────────────────────
# HARDEST VERBS (≥10 instances)
# ─────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"HARDEST VERBS — bottom 10 by accuracy (Viterbi, ≥10 instances)")
print(f"{'='*80}")
print(f"  {'Verb':<15} {'patB acc':>10} {'patC acc':>10} {'C-B':>8}  {'count':>6}")
print(f"  {'─'*15} {'─'*10} {'─'*10} {'─'*8}  {'─'*6}")
hard_verbs_b = [(v, info) for v, info in patB_err["verb_stats"].items() if info["total"] >= 10]
hard_verbs_b.sort(key=lambda x: x[1]["correct"] / x[1]["total"])
for v, info_b in hard_verbs_b[:10]:
    acc_b = info_b["correct"] / info_b["total"]
    info_c = patC_err["verb_stats"].get(v, {"correct": 0, "total": info_b["total"]})
    acc_c = info_c["correct"] / max(info_c["total"], 1)
    print(f"  {v:<15} {acc_b:>10.3f} {acc_c:>10.3f} {acc_c-acc_b:>+8.3f}  {info_b['total']:>6}")


# ─────────────────────────────────────────────────────
# BIO TRANSITION VALIDITY
# ─────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"BIO TRANSITION VALIDITY (lower = cleaner argmax outputs)")
print(f"{'='*80}")
for label, err in [("patB", patB_err), ("patC", patC_err)]:  1. File a uni-db follow-up asking if label disjunction :Foo|Bar in MATCH could be supported — that would close the gap on ABOUT and likely lift it to similar 3.9x speedup

    inv_a = err["invalid_transitions"]["argmax"]
    inv_v = err["invalid_transitions"]["viterbi"]
    tot = err["invalid_transitions"]["total"]
    print(f"  {label:<8} argmax invalid: {inv_a:5d}/{tot} ({100*inv_a/max(tot,1):.2f}%) | "
          f"viterbi invalid: {inv_v}/{tot} (Viterbi always 0%)")


# ─────────────────────────────────────────────────────
# VERDICT
# ─────────────────────────────────────────────────────
print(f"\n\n{'='*80}")
print("VERDICT — which Stage 2 pattern wins for xsmall v2?")
print(f"{'='*80}")
b_srl = patB_results["SRL_vit"]; c_srl = patC_results["SRL_vit"]
b_cls = patB_results["CLS"]; c_cls = patC_results["CLS"]

if b_srl > c_srl:
    srl_winner = "B"; srl_diff = b_srl - c_srl
else:
    srl_winner = "C"; srl_diff = c_srl - b_srl

if b_cls > c_cls:
    cls_winner = "B"; cls_diff = b_cls - c_cls
else:
    cls_winner = "C"; cls_diff = c_cls - b_cls

print(f"  SRL: Pattern {srl_winner} wins by {srl_diff:.4f} ({b_srl:.4f} vs {c_srl:.4f})")
print(f"  CLS: Pattern {cls_winner} wins by {cls_diff:.4f} ({b_cls:.4f} vs {c_cls:.4f})")
print(f"\n  Combined SRL+CLS:  Pattern B = {b_srl + b_cls:.4f},  Pattern C = {c_srl + c_cls:.4f}")
print(f"  Recommended for small/base: Pattern {'B' if (b_srl+b_cls) > (c_srl+c_cls) else 'C'}")
print(f"{'='*80}")

Loading eval data...

LOADING + EVALUATING: v2
  /content/drive/MyDrive/kniv-models/students_v2/deberta-v3-xsmall-v2/model.pt


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d


LOADING + EVALUATING: v2-patC
  /content/drive/MyDrive/kniv-models/students_v2/deberta-v3-xsmall-v2-patC/model.pt


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d



FULL 5-HEAD COMPARISON
  Head         v5 base       patB       patC    Teacher    B vs v5    C vs v5
  ────────── ─────────  ─────────  ─────────  ─────────  ─────────  ─────────
  POS           0.9710     0.9722     0.9722     0.9770    +0.0012    +0.0012
  NER           0.9830     0.9819     0.9825     0.9780    -0.0011    -0.0005
  DEP           0.9420     0.9395     0.9403     0.9440    -0.0025    -0.0017
  SRL           0.8750     0.8762     0.8885     0.9200    +0.0012    +0.0135
  SRL_vit       0.8770     0.8831     0.8906     0.9200    +0.0061    +0.0136
  CLS           0.9420     0.9420     0.9360     0.9510    +0.0000    -0.0060

SRL ERROR CATEGORIES (Viterbi)
  Category                                  v5     patB     patC     B-v5     C-v5
  ─────────────────────────────────── ──────── ──────── ──────── ──────── ────────
  Missed args (gold≠O, pred=O)             353      321      238      -32     -115
  Hallucinated (gold=O, pred≠O)            194      168      197      

In [ ]:
# Cell 11: Verify saved checkpoints on Drive

print("=" * 70)
print("CHECKPOINT INVENTORY")
print("=" * 70)

if not os.path.exists(STUDENT_DIR):
    print(f"  Student dir doesn't exist yet: {STUDENT_DIR}")
else:
    for entry in sorted(os.listdir(STUDENT_DIR)):
        path = f"{STUDENT_DIR}/{entry}"
        if os.path.isdir(path):
            print(f"\n  {entry}/")
            files = sorted(os.listdir(path))
            total_gb = 0
            for f in files:
                fp = f"{path}/{f}"
                size_mb = os.path.getsize(fp) / 1e6
                total_gb += size_mb / 1024
                print(f"    {f}: {size_mb:.1f} MB")
            print(f"    [total: {total_gb:.2f} GB]")

# Total Drive usage
total_size = sum(os.path.getsize(f"{STUDENT_DIR}/{d}/{f}")
                 for d in os.listdir(STUDENT_DIR) if os.path.isdir(f"{STUDENT_DIR}/{d}")
                 for f in os.listdir(f"{STUDENT_DIR}/{d}")) if os.path.exists(STUDENT_DIR) else 0
print(f"\n{'='*70}")
print(f"Total student checkpoints on Drive: {total_size/1e9:.2f} GB")
print(f"{'='*70}")

CHECKPOINT INVENTORY

  deberta-v3-xsmall-v2/
    model.pt: 303.5 MB
    model_s1.pt: 303.6 MB
    tokenizer.json: 8.3 MB
    tokenizer_config.json: 0.0 MB
    [total: 0.60 GB]

  deberta-v3-xsmall-v2-patC/
    model.pt: 303.5 MB
    model_s2a_best.pt: 303.6 MB
    tokenizer.json: 8.3 MB
    tokenizer_config.json: 0.0 MB
    [total: 0.60 GB]

Total student checkpoints on Drive: 1.23 GB


In [ ]:
# Cell 11: Run small student — Stage 1 + Pattern B + Pattern C

# Small (768d, 6 layers) — adjusted batch + LRs for larger model
print("=" * 70)
print("RUNNING SMALL STUDENT — full v2 pipeline")
print("=" * 70)

model_sm, tok_sm, out_sm = train_student(
    "microsoft/deberta-v3-small",
    suffix="v2",
    EPOCHS_S1=8,
    BATCH_SIZE=32,         # half of xsmall (~2x params)
    LR_ENC=1e-5,           # lower for larger pretrained model
    LR_HEADS=5e-4,
    USE_RDROP=True,
    USE_HIDDEN_DISTILL=True,
)

# Pattern B (joint sequential)
best_b_sm = stage2_finetune(model_sm, tok_sm, out_sm,
                            EPOCHS_S2=10, BATCH_SIZE=32,
                            LR_ENC=2e-6, LR_HEADS=5e-4)
print(f"\n✓ small Pattern B: {best_b_sm:.4f}")

# Pattern C (frozen-encoder CLS)
best_srl_c_sm, best_cls_c_sm = stage2_finetune_pattern_c(
    "microsoft/deberta-v3-small",
    suffix_in="v2", suffix_out="v2-patC",
    EPOCHS_S2A=8, EPOCHS_S2B=4, BATCH_SIZE=32,
    LR_ENC=2e-6, LR_HEADS=5e-4, LR_CLS_S2B=3e-4,
)
print(f"\n✓ small Pattern C: SRL={best_srl_c_sm:.4f}, CLS={best_cls_c_sm:.4f}")

RUNNING SMALL STUDENT — full v2 pipeline
TRAINING microsoft/deberta-v3-small v2 (v2)
  Stage 1: 8 epochs distillation
  Stage 2a: 8 epochs SRL gold+silver fine-tune
  R-Drop: True, Hidden distillation: True


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

...

  S2a E8 — SRL argmax: 0.8664, Viterbi: 0.8678

  Loading best SRL checkpoint (Vit 0.8808) for Stage 2b...

--- Stage 2b: CLS-only, encoder + SRL frozen (4 epochs) ---
  Trainable: 2.37M (CLS only), Frozen: 157.1M


  S2b E1 — SRL Vit (frozen, sanity): 0.8808, CLS: 0.9280
    Saved (SRL Vit: 0.8808, CLS: 0.9280)


  S2b E2 — SRL Vit (frozen, sanity): 0.8808, CLS: 0.9380
    Saved (SRL Vit: 0.8808, CLS: 0.9380)


  S2b E3 — SRL Vit (frozen, sanity): 0.8808, CLS: 0.9400
    Saved (SRL Vit: 0.8808, CLS: 0.9400)


  S2b E4 — SRL Vit (frozen, sanity): 0.8808, CLS: 0.9440
    Saved (SRL Vit: 0.8808, CLS: 0.9440)

PATTERN C COMPLETE — Best SRL: 0.8808, Best CLS: 0.9440
  Saved to: /content/drive/MyDrive/kniv-models/students_v2/deberta-v3-small-v2-patC

✓ small Pattern C: SRL=0.8808, CLS=0.9440


In [ ]:
# Cell 12: Run base student — Stage 1 + Pattern C (resumable)
# Skips Stage 1 if model_s1.pt already exists on Drive, so a kernel restart
# doesn't waste 5 hours redoing distillation. Goes straight to Pattern C if so.

import os

print("=" * 70)
print("RUNNING BASE STUDENT — full v2 pipeline (resumable)")
print("=" * 70)

s1_path = f"{STUDENT_DIR}/deberta-v3-base-v2/model_s1.pt"

if os.path.exists(s1_path):
    size_gb = os.path.getsize(s1_path) / 1e9
    print(f"\n✓ Stage 1 checkpoint already exists ({size_gb:.2f} GB)")
    print(f"  Path: {s1_path}")
    print(f"  Skipping Stage 1, going directly to Pattern C\n")
else:
    print(f"\n  Stage 1 checkpoint not found at {s1_path}")
    print(f"  Running full Stage 1 distillation (~5 hours)...\n")
    model_bs, tok_bs, out_bs = train_student(
        "microsoft/deberta-v3-base",
        suffix="v2",
        EPOCHS_S1=8,
        BATCH_SIZE=16,         # quarter of xsmall (~4x params and depth)
        LR_ENC=8e-6,
        LR_HEADS=3e-4,
        USE_RDROP=True,
        USE_HIDDEN_DISTILL=True,
    )

# Pattern C — independently loads from model_s1.pt, so this works whether
# Stage 1 just ran or was already on disk.
best_srl_c_bs, best_cls_c_bs = stage2_finetune_pattern_c(
    "microsoft/deberta-v3-base",
    suffix_in="v2", suffix_out="v2-patC",
    EPOCHS_S2A=8, EPOCHS_S2B=4, BATCH_SIZE=16,
    LR_ENC=1.5e-6, LR_HEADS=3e-4, LR_CLS_S2B=2e-4,
)
print(f"\n✓ base Pattern C: SRL={best_srl_c_bs:.4f}, CLS={best_cls_c_bs:.4f}")

RUNNING BASE STUDENT — full v2 pipeline (resumable)

✓ Stage 1 checkpoint already exists (0.81 GB)
  Path: /content/drive/MyDrive/kniv-models/students_v2/deberta-v3-base-v2/model_s1.pt
  Skipping Stage 1, going directly to Pattern C


PATTERN C: Sequential SRL-only → CLS-only (frozen encoder)
  Loading Stage 1 from: microsoft/deberta-v3-base-v2/model_s1.pt


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

  S2a E8 — SRL argmax: 0.8739, Viterbi: 0.8747

  Loading best SRL checkpoint (Vit 0.8856) for Stage 2b...

--- Stage 2b: CLS-only, encoder + SRL frozen (4 epochs) ---
  Trainable: 2.37M (CLS only)


  S2b E1 — SRL Vit (frozen): 0.8856, CLS: 0.9300  ★ BEST CLS (0.9300)


  S2b E2 — SRL Vit (frozen): 0.8856, CLS: 0.9380  ★ BEST CLS (0.9380)


  S2b E3 — SRL Vit (frozen): 0.8856, CLS: 0.9400  ★ BEST CLS (0.9400)


  S2b E4 — SRL Vit (frozen): 0.8856, CLS: 0.9460  ★ BEST CLS (0.9460)

PATTERN C COMPLETE — Best SRL: 0.8856, Best CLS: 0.9460
  Saved: model.pt (final), model_s2a_best.pt (best SRL), training_log.json
  → /content/drive/MyDrive/kniv-models/students_v2/deberta-v3-base-v2-patC

✓ base Pattern C: SRL=0.8856, CLS=0.9460


In [ ]:
# Cell 13: Final comparison across all sizes (xsmall + small + base) × (patB + patC)
# Self-contained — defines eval helpers inline so it runs after a kernel restart.

from collections import Counter, defaultdict

# ── Load eval data ──
print("Loading eval data...")
with open(f"{DATA}/ud_dev.json") as f: ud_dev = json.load(f)
with open(f"{DATA}/ner_spanmarker_dev.json") as f: ner_dev = json.load(f)
with open(f"{DATA}/srl_dev.json") as f: srl_dev = json.load(f)
with open(f"{DATA}/cls_sgd_mwoz_train.json") as f: cls_data = json.load(f)
random.seed(42); random.shuffle(cls_data); cls_dev_data = cls_data[:4000]


def evaluate_5_heads(model, tok, n_eval=500):
    """Run all 5-head eval. Returns dict of head accuracies + Viterbi for SRL."""
    model.eval()
    results = {}
    for task, data in [("POS", ud_dev), ("NER", ner_dev), ("DEP", ud_dev),
                        ("SRL", srl_dev), ("CLS", cls_dev_data)]:
        c, t, c_vit = 0, 0, 0
        with torch.no_grad():
            for ex in tqdm(data[:n_eval], desc=task, leave=False):
                if task in ("POS", "NER", "DEP"):
                    enc = tok(ex["words"], is_split_into_words=True, return_tensors="pt",
                              padding=True, truncation=True, max_length=128)
                    out = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                                torch.zeros(1, dtype=torch.long, device=device))
                    w2t = {}; prev = None
                    for k, wid in enumerate(enc.word_ids()):
                        if wid is not None and wid != prev: w2t[wid] = k
                        prev = wid
                    if task == "POS":
                        for wid, tidx in w2t.items():
                            if wid < len(ex["pos_tags"]):
                                if POS_LABELS[out[0][0, tidx].argmax().item()] == ex["pos_tags"][wid]: c += 1
                                t += 1
                    elif task == "NER":
                        for wid, tidx in w2t.items():
                            if wid < len(ex.get("ner_tags", [])):
                                if NER_LABELS[out[1][0, tidx].argmax().item()] == ex["ner_tags"][wid]: c += 1
                                t += 1
                    elif task == "DEP":
                        t2w = {v: k for k, v in w2t.items()}
                        for wid, tidx in w2t.items():
                            if wid >= len(ex["heads"]): continue
                            gold = ex["heads"][wid]
                            pred_wid = t2w.get(out[2][0, tidx].argmax().item(), -99)
                            if gold == -1:
                                if pred_wid == wid: c += 1
                            elif pred_wid == gold: c += 1
                            t += 1
                elif task == "SRL":
                    enc = tok(ex["words"], is_split_into_words=True, return_tensors="pt",
                              padding=True, truncation=True, max_length=128)
                    pt = 0; prev = None
                    for k, wid in enumerate(enc.word_ids()):
                        if wid is not None and wid != prev and wid == ex["predicate_idx"]: pt = k; break
                        prev = wid
                    out = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                                torch.tensor([pt], dtype=torch.long, device=device))
                    vi = []; prev = None
                    for k, wid in enumerate(enc.word_ids()):
                        if wid is not None and wid != prev and wid < len(ex["srl_tags"]): vi.append((k, wid))
                        prev = wid
                    if not vi: continue
                    valid_logits = out[4][0, [k for k, _ in vi]].cpu()
                    vit_path = viterbi_decode_srl(valid_logits)
                    for idx, (k, wid) in enumerate(vi):
                        gold = ex["srl_tags"][wid]
                        if SRL_TAGS[out[4][0, k].argmax().item()] == gold: c += 1
                        if SRL_TAGS[vit_path[idx]] == gold: c_vit += 1
                        t += 1
                elif task == "CLS":
                    text = ex.get("text", ""); pt = ex.get("prev_text")
                    if pt: enc = tok(pt, text, return_tensors="pt", padding=True, truncation=True, max_length=128)
                    else: enc = tok(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
                    out = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                                torch.zeros(1, dtype=torch.long, device=device))
                    if CLS_LABELS[out[5].argmax(-1).item()] == ex["cls_label"]: c += 1
                    t += 1
        results[task] = c / t if t > 0 else 0
        if task == "SRL":
            results["SRL_vit"] = c_vit / t if t > 0 else 0
    return results


def srl_error_analysis(model, tok, n_eval=500):
    """Detailed SRL error analysis. Returns dict with cats, span F1."""
    model.eval()
    cats = {"miss": 0, "halluc": 0, "boundary": 0, "role": 0, "total_errors": 0, "total_tokens": 0}
    span_g, span_p, span_m = 0, 0, 0

    def cat_err(g, p):
        if g == "O" and p != "O": return "halluc"
        if g != "O" and p == "O": return "miss"
        if g != "O" and p != "O":
            gp, gr = (g[0], g[2:]) if "-" in g else (g, g)
            pp, pr = (p[0], p[2:]) if "-" in p else (p, p)
            if gr == pr and gp != pp: return "boundary"
            return "role"
        return None

    def extract_spans(seq):
        spans = []; i = 0
        while i < len(seq):
            if seq[i].startswith("B-"):
                role = seq[i][2:]; start = i; i += 1
                while i < len(seq) and seq[i] == f"I-{role}": i += 1
                spans.append((role, start, i - 1))
            elif seq[i] == "V":
                spans.append(("V", i, i)); i += 1
            else: i += 1
        return spans

    with torch.no_grad():
        for ex in tqdm(srl_dev[:n_eval], desc="error analysis", leave=False):
            words = ex["words"]; pred_word_idx = ex["predicate_idx"]
            enc = tok(words, is_split_into_words=True, return_tensors="pt",
                      padding=True, truncation=True, max_length=128)
            pt = 0; prev = None
            for k, wid in enumerate(enc.word_ids()):
                if wid is not None and wid != prev and wid == pred_word_idx: pt = k; break
                prev = wid
            srl_logits = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                               torch.tensor([pt], dtype=torch.long, device=device))[4]
            vi = []; prev = None
            for k, wid in enumerate(enc.word_ids()):
                if wid is not None and wid != prev and wid < len(ex["srl_tags"]): vi.append((k, wid))
                prev = wid
            if not vi: continue
            valid_logits = srl_logits[0, [k for k, _ in vi]].cpu()
            vit_path = viterbi_decode_srl(valid_logits)
            gold_seq, pred_vit_seq = [], []
            for idx, (k, wid) in enumerate(vi):
                g = ex["srl_tags"][wid]
                p = SRL_TAGS[vit_path[idx]]
                gold_seq.append(g); pred_vit_seq.append(p)
                cats["total_tokens"] += 1
                if g != p:
                    cats["total_errors"] += 1
                    cat = cat_err(g, p)
                    if cat: cats[cat] += 1
            gs = set(extract_spans(gold_seq)); ps = set(extract_spans(pred_vit_seq))
            span_g += len(gs); span_p += len(ps); span_m += len(gs & ps)

    prec = span_m / span_p if span_p > 0 else 0
    rec = span_m / span_g if span_g > 0 else 0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0
    return {"cats": cats, "span_f1": (prec, rec, f1)}


def load_and_eval(student_name, suffix):
    """Load a v2 student variant and run full eval + SRL error analysis."""
    path = f"{STUDENT_DIR}/{student_name.split('/')[-1]}-{suffix}/model.pt"
    print(f"\n  Loading + evaluating: {student_name.split('/')[-1]}-{suffix}")
    tok = AutoTokenizer.from_pretrained(student_name)
    enc = AutoModel.from_pretrained(student_name).float()
    H = enc.config.hidden_size; NL = enc.config.num_hidden_layers + 1
    m = StudentCascadeV5(enc, H, NL).to(device)
    m.load_state_dict(torch.load(path, map_location="cpu"))
    results = evaluate_5_heads(m, tok)
    err = srl_error_analysis(m, tok)
    del m, enc; torch.cuda.empty_cache()
    return results, err


# ─────────────────────────────────────────────────────
# Run evaluation across all sizes × patterns
# ─────────────────────────────────────────────────────
print("=" * 80)
print("FINAL COMPARISON — all v2 students vs v5 baselines")
print("=" * 80)

all_results = {}

for size in ["xsmall", "small", "base"]:
    sn = f"microsoft/deberta-v3-{size}"
    for pat in ["v2", "v2-patC"]:
        path = f"{STUDENT_DIR}/deberta-v3-{size}-{pat}/model.pt"
        if not os.path.exists(path):
            print(f"  ✗ Skipping {size}-{pat}: {path} not found")
            continue
        results, err = load_and_eval(sn, pat)
        all_results[f"{size}-{pat}"] = (results, err)


# ── v5 baselines (from prior runs, for reference) ──
v5 = {
    "xsmall": {"POS": 0.971, "NER": 0.983, "DEP": 0.942, "SRL": 0.875, "SRL_vit": 0.877, "CLS": 0.942},
    "small":  {"POS": 0.973, "NER": 0.984, "DEP": 0.942, "SRL": 0.894, "SRL_vit": 0.897, "CLS": 0.954},
    "base":   {"POS": 0.974, "NER": 0.984, "DEP": 0.943, "SRL": 0.886, "SRL_vit": 0.892, "CLS": 0.960},
}
teacher = {"POS": 0.977, "NER": 0.978, "DEP": 0.944, "SRL": 0.920, "SRL_vit": 0.920, "CLS": 0.951}


# ── Master SRL Viterbi comparison ──
print(f"\n{'='*100}")
print(f"SRL VITERBI ACCURACY (key metric) — internal dev set")
print(f"{'='*100}")
print(f"  {'Size':<10} {'v5 base':>9} {'v2 patB':>9} {'v2 patC':>9} {'Teacher':>9} "
      f"{'B - v5':>9} {'C - v5':>9}")
print(f"  {'─'*10} {'─'*9} {'─'*9} {'─'*9} {'─'*9} {'─'*9} {'─'*9}")
for size in ["xsmall", "small", "base"]:
    base_v5 = v5[size]["SRL_vit"]
    b = all_results.get(f"{size}-v2", (None, None))[0]
    c = all_results.get(f"{size}-v2-patC", (None, None))[0]
    b_srl = b["SRL_vit"] if b else None
    c_srl = c["SRL_vit"] if c else None
    bs  = f"{b_srl:.4f}"      if b_srl is not None else "  --   "
    cs  = f"{c_srl:.4f}"      if c_srl is not None else "  --   "
    bvs = f"{b_srl-base_v5:+.4f}" if b_srl is not None else "   --   "
    cvs = f"{c_srl-base_v5:+.4f}" if c_srl is not None else "   --   "
    print(f"  {size:<10} {base_v5:>9.4f} {bs:>9} {cs:>9} {teacher['SRL_vit']:>9.4f} {bvs:>9} {cvs:>9}")
print(f"\n  Target: SRL ≥ 0.90")


# ── Full 5-head per size ──
for size in ["xsmall", "small", "base"]:
    print(f"\n{'='*100}")
    print(f"FULL 5-HEAD — {size}")
    print(f"{'='*100}")
    print(f"  {'Head':<10} {'v5':>9} {'v2 patB':>9} {'v2 patC':>9} {'Teacher':>9}")
    print(f"  {'─'*10} {'─'*9} {'─'*9} {'─'*9} {'─'*9}")
    b = all_results.get(f"{size}-v2", (None, None))[0]
    c = all_results.get(f"{size}-v2-patC", (None, None))[0]
    for h in ["POS", "NER", "DEP", "SRL", "SRL_vit", "CLS"]:
        v5v = v5[size][h]
        bv = f"{b[h]:.4f}" if b else "  --  "
        cv = f"{c[h]:.4f}" if c else "  --  "
        tv = teacher[h]
        print(f"  {h:<10} {v5v:>9.4f} {bv:>9} {cv:>9} {tv:>9.4f}")


# ── SRL error categories — Pattern C across sizes ──
print(f"\n{'='*100}")
print(f"SRL ERROR CATEGORIES — Pattern C across sizes")
print(f"{'='*100}")
print(f"  {'Category':<35} {'xsmall patC':>12} {'small patC':>12} {'base patC':>12}")
print(f"  {'─'*35} {'─'*12} {'─'*12} {'─'*12}")
for cat in ["miss", "halluc", "boundary", "role", "total_errors"]:
    label = {"miss": "Missed args", "halluc": "Hallucinated",
             "boundary": "Boundary errors", "role": "Role confusion",
             "total_errors": "TOTAL ERRORS"}[cat]
    vals = []
    for size in ["xsmall", "small", "base"]:
        e = all_results.get(f"{size}-v2-patC", (None, None))[1]
        vals.append(f"{e['cats'][cat]}" if e else "--")
    print(f"  {label:<35} {vals[0]:>12} {vals[1]:>12} {vals[2]:>12}")


# ── Span-level F1 — Pattern C across sizes ──
print(f"\n{'='*100}")
print(f"SPAN-LEVEL F1 — Pattern C across sizes")
print(f"{'='*100}")
print(f"  {'Size':<10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"  {'─'*10} {'─'*10} {'─'*10} {'─'*10}")
for size in ["xsmall", "small", "base"]:
    e = all_results.get(f"{size}-v2-patC", (None, None))[1]
    if e:
        p, r, f = e["span_f1"]
        print(f"  {size:<10} {p:>10.4f} {r:>10.4f} {f:>10.4f}")


# ── Verdict ──
print(f"\n\n{'='*100}")
print("FINAL VERDICT")
print(f"{'='*100}")
hit_target = []
for size in ["xsmall", "small", "base"]:
    c = all_results.get(f"{size}-v2-patC", (None, None))[0]
    if c and c["SRL_vit"] >= 0.90:
        hit_target.append(size)
        print(f"  ✓ {size} v2 patC: SRL Vit = {c['SRL_vit']:.4f}  (target hit)")
    elif c:
        print(f"  ✗ {size} v2 patC: SRL Vit = {c['SRL_vit']:.4f}  (gap to 0.90: {0.90-c['SRL_vit']:+.4f})")

print(f"\n  Sizes hitting 0.90 SRL target: {len(hit_target)}/3 ({', '.join(hit_target) if hit_target else 'none'})")
print(f"{'='*100}")

Loading eval data...
FINAL COMPARISON — all v2 students vs v5 baselines

  Loading + evaluating: deberta-v3-xsmall-v2


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/241M [00:00<?, ?B/s]

...

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✗ Skipping base-v2: /content/drive/MyDrive/kniv-models/students_v2/deberta-v3-base-v2/model.pt not found

  Loading + evaluating: deberta-v3-base-v2-patC


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



SRL VITERBI ACCURACY (key metric) — internal dev set
  Size         v5 base   v2 patB   v2 patC   Teacher    B - v5    C - v5
  ────────── ───────── ───────── ───────── ───────── ───────── ─────────
  xsmall        0.8770    0.8831    0.8906    0.9200   +0.0061   +0.0136
  small         0.8970    0.8825    0.8884    0.9200   -0.0145   -0.0086
  base          0.8920     --       0.8939    0.9200     --      +0.0019

  Target: SRL ≥ 0.90

FULL 5-HEAD — xsmall
  Head              v5   v2 patB   v2 patC   Teacher
  ────────── ───────── ───────── ───────── ─────────
  POS           0.9710    0.9722    0.9722    0.9770
  NER           0.9830    0.9819    0.9825    0.9780
  DEP           0.9420    0.9395    0.9403    0.9440
  SRL           0.8750    0.8762    0.8885    0.9200
  SRL_vit       0.8770    0.8831    0.8906    0.9200
  CLS           0.9420    0.9420    0.9360    0.9510

FULL 5-HEAD — small
  Head              v5   v2 patB   v2 patC   Teacher
  ────────── ───────── ───────── ──────

In [ ]:
# Cell 15: StudentCascadeUniversal — works with any HF encoder
#
# Same architecture as StudentCascadeV5 (same heads, same cascade flow),
# but uses get_input_embeddings() + inputs_embeds= for predicate injection
# so it works with ModernBERT, NeoBERT, DeBERTa, and any standard HF encoder.
#
# Key differences from StudentCascadeV5:
#   - Predicate injection always uses inputs_embeds= path (universal)
#   - hidden states from output_hidden_states=True work the same way
#   - Everything downstream (POS, NER, DEP, SRL, CLS heads) is identical
#
# Performance is essentially identical to v5 on DeBERTa; the inputs_embeds
# path is the standard HF API and adds no measurable overhead.

class StudentCascadeUniversal(nn.Module):
    """v5 cascade with universal HF encoder API. Predicate injection via inputs_embeds."""
    def __init__(self, encoder, H, num_layers):
        super().__init__()
        self.encoder = encoder
        self.pred_embedding = nn.Embedding(2, H)
        nn.init.zeros_(self.pred_embedding.weight)
        NL = num_layers

        # POS
        self.pos_sm = ScalarMix(NL)
        self.pos_head = nn.Linear(H, N_POS)

        # NER
        self.ner_sm = ScalarMix(NL)
        lstm_h = max(H // 4, 64)
        self.ner_lstm = nn.LSTM(H, lstm_h, bidirectional=True, batch_first=True)
        self.ner_proj = nn.Linear(lstm_h * 2, H)
        self.ner_head = nn.Sequential(nn.LayerNorm(H+N_POS), nn.Linear(H+N_POS, H),
                                      nn.GELU(), nn.Dropout(0.1), nn.Linear(H, N_NER))

        # DEP — BiLSTM before biaffine
        self.dep_sm = ScalarMix(NL)
        in_dim = H + N_POS + N_NER
        self.dep_proj = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, H), nn.GELU())
        dep_lstm_h = max(H // 4, 64)
        self.dep_lstm = nn.LSTM(H, dep_lstm_h, bidirectional=True, batch_first=True)
        self.dep_lstm_proj = nn.Linear(dep_lstm_h * 2, H)
        arc_dim, label_dim = max(H//2, 64), max(H//8, 32)
        self.dep_biaff = BiaffineDEPHead(H, arc_dim, label_dim, N_DEP)

        # SRL
        srl_input_dim = H * 4 + N_POS + N_DEP
        self.srl_sm = ScalarMix(NL)
        self.srl_interaction_proj = nn.Sequential(
            nn.LayerNorm(srl_input_dim), nn.Linear(srl_input_dim, H),
            nn.GELU(), nn.Dropout(0.1))
        srl_lstm_h = max(H // 2, 96)
        self.srl_lstm = nn.LSTM(H, srl_lstm_h, bidirectional=True, batch_first=True)
        self.srl_proj = nn.Linear(srl_lstm_h * 2, H)
        self.srl_head = nn.Sequential(nn.Dropout(0.1), nn.Linear(H, H), nn.GELU(),
                                      nn.Dropout(0.1), nn.Linear(H, N_SRL))

        # CLS
        self.cls_sm = ScalarMix(NL)
        self.cls_pool = AttentionPool(H)
        self.cls_head = nn.Sequential(nn.LayerNorm(H), nn.Linear(H, H * 2), nn.GELU(),
                                      nn.Dropout(0.1), nn.Linear(H * 2, H), nn.GELU(),
                                      nn.Dropout(0.1), nn.Linear(H, N_CLS))

        # Hidden state distillation projections (one per teacher layer we track)
        self.distill_projections = nn.ModuleDict({
            f"l{l}": nn.Linear(H, H_TEACHER, bias=False) for l in TEACHER_LAYERS
        })
        for proj in self.distill_projections.values():
            nn.init.xavier_uniform_(proj.weight, gain=0.5)

    def forward(self, input_ids, attention_mask, predicate_idx, return_hidden_states=False):
        B, S = input_ids.size()

        # Universal predicate injection: get embeddings, add predicate marker, forward via inputs_embeds
        emb_layer = self.encoder.get_input_embeddings()
        emb = emb_layer(input_ids)
        indicator = torch.zeros(B, S, dtype=torch.long, device=input_ids.device)
        indicator.scatter_(1, predicate_idx.unsqueeze(1), 1)
        emb = emb + self.pred_embedding(indicator)

        enc_out = self.encoder(inputs_embeds=emb, attention_mask=attention_mask,
                               output_hidden_states=True)
        layers = list(enc_out.hidden_states)

        # POS
        pos_logits = self.pos_head(self.pos_sm(layers))
        pos_p = torch.softmax(pos_logits, -1)

        # NER
        ner_h = self.ner_sm(layers)
        lo, _ = self.ner_lstm(ner_h)
        ner_logits = self.ner_head(torch.cat([self.ner_proj(lo) + ner_h, pos_p.detach()], -1))
        ner_p = torch.softmax(ner_logits, -1)

        # DEP
        dep_h = self.dep_sm(layers)
        dep_proj = self.dep_proj(torch.cat([dep_h, pos_p.detach(), ner_p.detach()], -1))
        dep_lo, _ = self.dep_lstm(dep_proj)
        dep_adapted = self.dep_lstm_proj(dep_lo) + dep_proj
        arc_scores, label_scores = self.dep_biaff(dep_adapted)

        # DEP cascade features for SRL
        pred_heads = arc_scores.argmax(dim=-1)
        ph = pred_heads.clamp(min=0).unsqueeze(-1).unsqueeze(-1).expand(B, S, 1, N_DEP)
        dep_label_at_head = label_scores.gather(2, ph).squeeze(2)
        dep_rel_probs = torch.softmax(dep_label_at_head, dim=-1).detach()

        # SRL
        srl_h = self.srl_sm(layers)
        pred_idx_expanded = predicate_idx.view(B, 1, 1).expand(B, 1, srl_h.size(-1))
        h_pred = srl_h.gather(1, pred_idx_expanded).expand_as(srl_h)
        h_i = srl_h
        srl_features = torch.cat([
            h_i, h_pred, h_i * h_pred, (h_i - h_pred).abs(),
            pos_p.detach(), dep_rel_probs,
        ], dim=-1)
        srl_projected = self.srl_interaction_proj(srl_features)
        srl_lo, _ = self.srl_lstm(srl_projected)
        srl_adapted = self.srl_proj(srl_lo) + srl_projected
        srl_logits = self.srl_head(srl_adapted)

        # CLS
        cls_h = self.cls_sm(layers)
        cls_logits = self.cls_head(self.cls_pool(cls_h, attention_mask))

        if return_hidden_states:
            n_student_layers = len(layers) - 1
            mapped_hidden = {}
            for tl in TEACHER_LAYERS:
                student_layer_idx = max(1, round(n_student_layers * tl / 24))
                mapped_hidden[tl] = layers[student_layer_idx]
            return pos_logits, ner_logits, arc_scores, label_scores, srl_logits, cls_logits, mapped_hidden
        return pos_logits, ner_logits, arc_scores, label_scores, srl_logits, cls_logits


print("StudentCascadeUniversal defined.")
print("  Works with: DeBERTa-v3-*, ModernBERT, NeoBERT, and any HF encoder")
print("  Predicate injection via get_input_embeddings() + inputs_embeds=")

StudentCascadeUniversal defined.
  Works with: DeBERTa-v3-*, ModernBERT, NeoBERT, and any HF encoder
  Predicate injection via get_input_embeddings() + inputs_embeds=


In [ ]:
# Cell 16: Training pipeline for ModernBERT/NeoBERT — uses StudentCascadeUniversal
#
# Mirrors train_student() and stage2_finetune_pattern_c() but with the universal
# cascade class so it works with any HF encoder.

def train_universal_student(STUDENT_NAME, suffix="v2-universal",
                            EPOCHS_S1=8, BATCH_SIZE=32,
                            LR_ENC=1e-5, LR_HEADS=5e-4,
                            S1_WARMUP=0.06,
                            USE_RDROP=True, USE_HIDDEN_DISTILL=True,
                            trust_remote_code=True,
                            loss_weights=None):
    """Stage 1 distillation for any HF encoder. Same losses as train_student()."""
    if loss_weights is None:
        loss_weights = {
            "alpha_hard": 0.5, "beta_kl": 1.0, "gamma_hidden": 0.3, "delta_rdrop": 0.2,
            "dep_arc": 1.5, "dep_rel": 0.5,
            "ner_mult": 1.5, "srl_mult": 1.0, "cls_mult": 1.0,
            "hidden_mult_pred": 0.5,
        }

    print("=" * 70)
    print(f"TRAINING UNIVERSAL STUDENT: {STUDENT_NAME}")
    print(f"  Stage 1: {EPOCHS_S1} epochs distillation")
    print(f"  R-Drop: {USE_RDROP}, Hidden distillation: {USE_HIDDEN_DISTILL}")
    print("=" * 70)

    student_tok = AutoTokenizer.from_pretrained(STUDENT_NAME, trust_remote_code=trust_remote_code)
    student_enc = AutoModel.from_pretrained(STUDENT_NAME, trust_remote_code=trust_remote_code).float()
    H = student_enc.config.hidden_size
    NL = student_enc.config.num_hidden_layers + 1

    model = StudentCascadeUniversal(student_enc, H, NL).to(device)
    n_total = sum(p.numel() for p in model.parameters())
    n_enc = sum(p.numel() for p in model.encoder.parameters())
    n_distill_proj = sum(p.numel() for p in model.distill_projections.parameters())
    print(f"  Encoder: {H}d, {NL-1} layers, {n_enc/1e6:.0f}M")
    print(f"  Total: {n_total/1e6:.0f}M (heads: {(n_total-n_enc-n_distill_proj)/1e6:.1f}M, "
          f"distill projections: {n_distill_proj/1e6:.1f}M)")

    student_out = f"{STUDENT_DIR}/{STUDENT_NAME.split('/')[-1]}-{suffix}"
    os.makedirs(student_out, exist_ok=True)

    print(f"\n--- Stage 1: Distillation ({EPOCHS_S1} epochs) ---")
    shard_paths = sorted([f"{DISTILL_DIR}/{f}" for f in os.listdir(DISTILL_DIR)
                          if f.endswith(".parquet") and f.startswith("shard_")])
    dataset = DistillationDatasetV2(shard_paths, student_tok)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=collate, num_workers=2, pin_memory=True)

    enc_params = list(model.encoder.parameters()) + list(model.pred_embedding.parameters())
    head_params = [p for n, p in model.named_parameters()
                   if not n.startswith("encoder.") and not n.startswith("pred_embedding.")]
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENC},
        {"params": head_params, "lr": LR_HEADS}
    ], weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(EPOCHS_S1*len(loader)*S1_WARMUP), EPOCHS_S1*len(loader))
    T = 3.0; ALPHA = 0.7
    w = loss_weights

    def dl(s, t, v, T):
        sl = s.view(-1, s.size(-1))[v]; tl = t.view(-1, t.size(-1))[v]
        if sl.numel() == 0: return torch.tensor(0.0, device=device)
        return F.kl_div(F.log_softmax(sl/T, -1), F.softmax(tl/T, -1), reduction="batchmean") * T**2

    for epoch in range(EPOCHS_S1):
        model.train(); running = {"total": 0, "kl": 0, "hard": 0, "hidden": 0, "rdrop": 0, "n": 0}
        for batch in tqdm(loader, desc=f"S1 E{epoch+1}/{EPOCHS_S1}"):
            ids = batch["input_ids"].to(device); mask = batch["attention_mask"].to(device)
            pred = batch["predicate_idx"].to(device); valid = batch["valid"].to(device)
            B, S = ids.size(); fv = valid.view(B*S)

            out1 = model(ids, mask, pred, return_hidden_states=USE_HIDDEN_DISTILL)
            if USE_HIDDEN_DISTILL:
                s_pos, s_ner, s_arc, s_lab, s_srl, s_cls, student_hidden = out1
            else:
                s_pos, s_ner, s_arc, s_lab, s_srl, s_cls = out1

            kl_pos = dl(s_pos, batch["pos_t"].to(device), fv, T)
            kl_ner = dl(s_ner, batch["ner_t"].to(device), fv, T)
            sv = batch["srl_valid"].to(device)
            srl_mask = (valid & sv.unsqueeze(1)).view(B*S)
            kl_srl = dl(s_srl, batch["srl_t"].to(device), srl_mask, T) if srl_mask.any() else torch.tensor(0.0, device=device)
            tps = max(valid.sum().float().item()/B, 1.0)
            cls_ones = torch.ones(B, dtype=torch.bool, device=device)
            kl_cls = dl(s_cls, batch["cls_t"].to(device), cls_ones, T) * tps

            hard_pos = F.cross_entropy(s_pos.view(-1, N_POS), batch["pos_h"].to(device).view(-1), ignore_index=-100)
            hard_ner = F.cross_entropy(s_ner.view(-1, N_NER), batch["ner_h"].to(device).view(-1), ignore_index=-100)
            hard_srl = F.cross_entropy(s_srl.view(-1, N_SRL), batch["srl_hard"].to(device).view(-1), ignore_index=-100) if srl_mask.any() else torch.tensor(0.0, device=device)
            hard_cls = F.cross_entropy(s_cls, batch["cls_t"].to(device).argmax(-1))

            dh = batch["dep_head_hard"].to(device)
            dep_arc_loss = F.cross_entropy(s_arc.view(B*S, S), dh.view(B*S), ignore_index=-100)
            ph = dh.clamp(0).unsqueeze(-1).unsqueeze(-1).expand(B, S, 1, N_DEP)
            gl = s_lab.gather(2, ph).squeeze(2)
            dep_rel_loss = F.cross_entropy(gl.view(B*S, N_DEP),
                                           batch["dep_rel_hard"].to(device).view(B*S),
                                           ignore_index=-100)

            if USE_HIDDEN_DISTILL:
                hidden_loss = torch.tensor(0.0, device=device)
                for tl in TEACHER_LAYERS:
                    student_proj = model.distill_projections[f"l{tl}"](student_hidden[tl])
                    teacher_h = batch[f"teacher_hidden_l{tl}"].to(device)
                    sh = student_proj.view(-1, H_TEACHER)[fv]
                    th = teacher_h.view(-1, H_TEACHER)[fv]
                    if sh.numel() > 0:
                        sh = F.layer_norm(sh, [H_TEACHER]); th = F.layer_norm(th, [H_TEACHER])
                        hidden_loss = hidden_loss + F.mse_loss(sh, th)
                if srl_mask.any():
                    student_last = student_hidden[max(TEACHER_LAYERS)]
                    student_last_proj = model.distill_projections[f"l{max(TEACHER_LAYERS)}"](student_last)
                    sh = student_last_proj.view(-1, H_TEACHER)[srl_mask]
                    th = batch["teacher_pred_hidden"].to(device).view(-1, H_TEACHER)[srl_mask]
                    if sh.numel() > 0:
                        sh = F.layer_norm(sh, [H_TEACHER]); th = F.layer_norm(th, [H_TEACHER])
                        hidden_loss = hidden_loss + w["hidden_mult_pred"] * F.mse_loss(sh, th)
            else:
                hidden_loss = torch.tensor(0.0, device=device)

            if USE_RDROP:
                out2 = model(ids, mask, pred)
                _, s2_ner, _, _, s2_srl, s2_cls = out2
                def rd_kl(l1, l2, vmask, tt=1.0):
                    a = l1.view(-1, l1.size(-1))[vmask]; b = l2.view(-1, l2.size(-1))[vmask]
                    if a.numel() == 0: return torch.tensor(0.0, device=device)
                    p1 = F.log_softmax(a/tt, -1); p2 = F.log_softmax(b/tt, -1)
                    return 0.5 * (F.kl_div(p1, p2, log_target=True, reduction="batchmean")
                                  + F.kl_div(p2, p1, log_target=True, reduction="batchmean")) * tt**2
                rdrop_total = (rd_kl(s_ner, s2_ner, fv)
                               + (rd_kl(s_srl, s2_srl, srl_mask) if srl_mask.any() else 0)
                               + rd_kl(s_cls.unsqueeze(1), s2_cls.unsqueeze(1), cls_ones))
            else:
                rdrop_total = torch.tensor(0.0, device=device)

            soft_total = (1.0*kl_pos + w["ner_mult"]*kl_ner + w["srl_mult"]*kl_srl + w["cls_mult"]*kl_cls
                          + w["dep_arc"]*dep_arc_loss + w["dep_rel"]*dep_rel_loss)
            hard_total = (1.0*hard_pos + w["ner_mult"]*hard_ner + w["srl_mult"]*hard_srl + w["cls_mult"]*hard_cls)
            loss = (ALPHA * soft_total + (1-ALPHA) * w["alpha_hard"] * hard_total
                    + w["gamma_hidden"] * hidden_loss + w["delta_rdrop"] * rdrop_total)

            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            running["total"] += loss.item(); running["kl"] += soft_total.item()
            running["hard"] += hard_total.item()
            running["hidden"] += hidden_loss.item() if USE_HIDDEN_DISTILL else 0
            running["rdrop"] += rdrop_total.item() if USE_RDROP else 0
            running["n"] += 1
        n = running["n"]
        print(f"  S1 E{epoch+1} — Loss {running['total']/n:.3f}  "
              f"(kl {running['kl']/n:.3f}, hard {running['hard']/n:.3f}, "
              f"hidden {running['hidden']/n:.3f}, rdrop {running['rdrop']/n:.3f})")

    torch.save(model.state_dict(), f"{student_out}/model_s1.pt")
    print("Stage 1 done.\n")
    return model, student_tok, student_out


def stage2_universal_pattern_c(student_name, suffix_in="v2-universal", suffix_out="v2-universal-patC",
                               EPOCHS_S2A=8, EPOCHS_S2B=4, BATCH_SIZE=32,
                               LR_ENC=2e-6, LR_HEADS=5e-4, LR_CLS_S2B=3e-4,
                               SILVER_LIMIT=80000, trust_remote_code=True):
    """Pattern C Stage 2 for any HF encoder — same recipe as DeBERTa version."""
    print(f"\n{'='*70}")
    print(f"PATTERN C UNIVERSAL: {student_name}")
    print(f"  Loading Stage 1 from: {student_name}-{suffix_in}/model_s1.pt")
    print(f"{'='*70}")

    student_tok = AutoTokenizer.from_pretrained(student_name, trust_remote_code=trust_remote_code)
    student_enc = AutoModel.from_pretrained(student_name, trust_remote_code=trust_remote_code).float()
    H = student_enc.config.hidden_size; NL = student_enc.config.num_hidden_layers + 1
    model = StudentCascadeUniversal(student_enc, H, NL).to(device)

    s1_path = f"{STUDENT_DIR}/{student_name.split('/')[-1]}-{suffix_in}/model_s1.pt"
    model.load_state_dict(torch.load(s1_path, map_location="cpu"))
    print(f"  ✓ Loaded Stage 1 checkpoint")

    student_out = f"{STUDENT_DIR}/{student_name.split('/')[-1]}-{suffix_out}"
    os.makedirs(student_out, exist_ok=True)

    with open(f"{DATA}/srl_train.json") as f: srl_gold = json.load(f)
    with open(f"{DATA}/srl_dev.json") as f: srl_dev = json.load(f)
    shard_paths = sorted([f"{DISTILL_DIR}/{f}" for f in os.listdir(DISTILL_DIR)
                          if f.endswith(".parquet") and f.startswith("shard_")])
    teacher_silver = extract_teacher_srl_silver(shard_paths)
    random.seed(42)
    if SILVER_LIMIT and len(teacher_silver) > SILVER_LIMIT:
        teacher_silver = random.sample(teacher_silver, SILVER_LIMIT)
    srl_combined = teacher_silver + srl_gold * 2
    random.shuffle(srl_combined)
    print(f"  SRL: {len(srl_combined):,}")

    with open(f"{DATA}/cls_sgd_mwoz_train.json") as f: ca = json.load(f)
    random.seed(42); random.shuffle(ca); cls_dev_data = ca[:4000]; cls_train = ca[4000:]
    print(f"  CLS: {len(cls_train):,} train")

    class SRLDs(Dataset):
        def __init__(s, e, tok): s.e, s.tok = e, tok
        def __len__(s): return len(s.e)
        def __getitem__(s, i):
            ex = s.e[i]; enc = s.tok(ex["words"], is_split_into_words=True, max_length=128,
                                     padding="max_length", truncation=True, return_tensors="pt")
            pt, al, prev = 0, [], None
            for k, wid in enumerate(enc.word_ids()):
                if wid is None: al.append(-100)
                elif wid != prev:
                    al.append(srl_map.get(ex["srl_tags"][wid] if wid < len(ex["srl_tags"]) else "O", 0))
                    if wid == ex["predicate_idx"]: pt = k
                else: al.append(-100)
                prev = wid
            return {"input_ids": enc["input_ids"].squeeze(0),
                    "attention_mask": enc["attention_mask"].squeeze(0),
                    "labels": torch.tensor(al, dtype=torch.long),
                    "predicate_idx": torch.tensor(pt, dtype=torch.long)}

    class CLSDs(Dataset):
        def __init__(s, e, tok): s.e, s.tok = e, tok
        def __len__(s): return len(s.e)
        def __getitem__(s, i):
            ex = s.e[i]; t = ex.get("text",""); pt = ex.get("prev_text")
            if pt: enc = s.tok(pt, t, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
            else: enc = s.tok(t, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
            return {"input_ids": enc["input_ids"].squeeze(0),
                    "attention_mask": enc["attention_mask"].squeeze(0),
                    "labels": torch.tensor(cls_map[ex["cls_label"]], dtype=torch.long)}

    srl_ld = DataLoader(SRLDs(srl_combined, student_tok), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    cls_ld = DataLoader(CLSDs(cls_train, student_tok), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)

    # Stage 2a: SRL-only
    print(f"\n--- Stage 2a: SRL-only ({EPOCHS_S2A} epochs) ---")
    for p in model.parameters(): p.requires_grad = False
    for p in model.encoder.parameters(): p.requires_grad = True
    for p in model.pred_embedding.parameters(): p.requires_grad = True
    srl_modules = [model.srl_sm, model.srl_interaction_proj, model.srl_lstm, model.srl_proj, model.srl_head,
                   model.dep_sm, model.dep_proj, model.dep_lstm, model.dep_lstm_proj, model.dep_biaff,
                   model.pos_sm, model.pos_head,
                   model.ner_sm, model.ner_lstm, model.ner_proj, model.ner_head]
    for m in srl_modules:
        for p in m.parameters(): p.requires_grad = True

    enc_params = list(model.encoder.parameters()) + list(model.pred_embedding.parameters())
    head_params = [p for n, p in model.named_parameters()
                   if p.requires_grad and not n.startswith("encoder.") and not n.startswith("pred_embedding.")]
    optimizer = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENC},
                                   {"params": head_params, "lr": LR_HEADS}], weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(EPOCHS_S2A*len(srl_ld)*0.1), EPOCHS_S2A*len(srl_ld))
    best_srl = 0.0

    for epoch in range(EPOCHS_S2A):
        model.train()
        for b in tqdm(srl_ld, desc=f"S2a-SRL E{epoch+1}", leave=False):
            o = model(b["input_ids"].to(device), b["attention_mask"].to(device), b["predicate_idx"].to(device))[4]
            loss = F.cross_entropy(o.view(-1, N_SRL), b["labels"].to(device).view(-1), ignore_index=-100)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
        model.eval(); sc_arg, sc_vit, st = 0, 0, 0
        with torch.no_grad():
            for ex in srl_dev[:300]:
                enc = student_tok(ex["words"], is_split_into_words=True, return_tensors="pt", padding=True, truncation=True, max_length=128)
                pt = 0; prev = None
                for k, wid in enumerate(enc.word_ids()):
                    if wid is not None and wid != prev and wid == ex["predicate_idx"]: pt = k; break
                    prev = wid
                o = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                          torch.tensor([pt], dtype=torch.long, device=device))[4]
                vi = []; prev = None
                for k, wid in enumerate(enc.word_ids()):
                    if wid is not None and wid != prev and wid < len(ex["srl_tags"]): vi.append((k, wid))
                    prev = wid
                if not vi: continue
                valid_logits = o[0, [k for k, _ in vi]].cpu()
                vit_path = viterbi_decode_srl(valid_logits)
                for idx, (k, wid) in enumerate(vi):
                    gold = ex["srl_tags"][wid]
                    if SRL_TAGS[o[0, k].argmax().item()] == gold: sc_arg += 1
                    if SRL_TAGS[vit_path[idx]] == gold: sc_vit += 1
                    st += 1
        srl_arg = sc_arg/st if st > 0 else 0; srl_vit = sc_vit/st if st > 0 else 0
        print(f"  S2a E{epoch+1} — SRL argmax: {srl_arg:.4f}, Viterbi: {srl_vit:.4f}")
        if srl_vit > best_srl:
            best_srl = srl_vit
            torch.save(model.state_dict(), f"{student_out}/model_s2a_best.pt")
            print(f"    ★ BEST SRL ({best_srl:.4f})")

    print(f"\n  Loading best SRL checkpoint (Vit {best_srl:.4f}) for Stage 2b...")
    model.load_state_dict(torch.load(f"{student_out}/model_s2a_best.pt", map_location="cpu"))

    # Stage 2b: CLS-only
    print(f"\n--- Stage 2b: CLS-only ({EPOCHS_S2B} epochs) ---")
    for p in model.parameters(): p.requires_grad = False
    for m in [model.cls_sm, model.cls_pool, model.cls_head]:
        for p in m.parameters(): p.requires_grad = True

    cls_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(cls_params, lr=LR_CLS_S2B, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(EPOCHS_S2B*len(cls_ld)*0.1), EPOCHS_S2B*len(cls_ld))
    best_cls = 0.0

    for epoch in range(EPOCHS_S2B):
        model.train()
        model.encoder.eval()
        for b in tqdm(cls_ld, desc=f"S2b-CLS E{epoch+1}", leave=False):
            o = model(b["input_ids"].to(device), b["attention_mask"].to(device),
                      torch.zeros(b["input_ids"].size(0), dtype=torch.long, device=device))[5]
            loss = F.cross_entropy(o, b["labels"].to(device))
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
        model.eval()
        cc, ct = 0, 0
        with torch.no_grad():
            for ex in cls_dev_data[:500]:
                text = ex.get("text", ""); pt = ex.get("prev_text")
                if pt: enc = student_tok(pt, text, return_tensors="pt", padding=True, truncation=True, max_length=128)
                else: enc = student_tok(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
                o = model(enc["input_ids"].to(device), enc["attention_mask"].to(device),
                          torch.zeros(1, dtype=torch.long, device=device))[5]
                if CLS_LABELS[o.argmax(-1).item()] == ex["cls_label"]: cc += 1
                ct += 1
        cls_a = cc/ct if ct > 0 else 0
        print(f"  S2b E{epoch+1} — CLS: {cls_a:.4f}")
        if cls_a > best_cls:
            best_cls = cls_a
            torch.save(model.state_dict(), f"{student_out}/model.pt")
            student_tok.save_pretrained(student_out)
            print(f"    ★ BEST CLS ({best_cls:.4f})")

    print(f"\n{'='*70}")
    print(f"UNIVERSAL PATTERN C COMPLETE — Best SRL: {best_srl:.4f}, Best CLS: {best_cls:.4f}")
    print(f"  → {student_out}")
    print(f"{'='*70}")
    return best_srl, best_cls


print("Universal training functions defined:")
print("  train_universal_student(STUDENT_NAME, ...) — Stage 1 distillation")
print("  stage2_universal_pattern_c(student_name, ...) — Pattern C Stage 2")

Universal training functions defined:
  train_universal_student(STUDENT_NAME, ...) — Stage 1 distillation
  stage2_universal_pattern_c(student_name, ...) — Pattern C Stage 2


In [ ]:
!pip install xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.3 MB/s eta 0:00:00


In [ ]:
# Cell 14: Smoke test — ModernBERT and NeoBERT compatibility check
#
# Verifies that we can:
#   1. Load each encoder
#   2. Identify hidden_size, num_layers, and how to inject predicate embedding
#   3. Run a forward pass with output_hidden_states=True
# without committing to training.

from transformers import AutoModel, AutoTokenizer
import torch

CANDIDATES = [
    ("answerdotai/ModernBERT-base", "ModernBERT-base (149M)"),
    ("chandar-lab/NeoBERT",          "NeoBERT (250M)"),
]

print("=" * 70)
print("ENCODER COMPATIBILITY CHECK")
print("=" * 70)

for model_id, label in CANDIDATES:
    print(f"\n--- {label} ({model_id}) ---")
    try:
        tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        enc = AutoModel.from_pretrained(model_id, trust_remote_code=True).float()
    except Exception as e:
        print(f"  ✗ Load failed: {type(e).__name__}: {e}")
        continue

    H = enc.config.hidden_size
    NL = enc.config.num_hidden_layers
    n_params = sum(p.numel() for p in enc.parameters()) / 1e6

    has_emb_attr = hasattr(enc, "embeddings")
    has_encoder_attr = hasattr(enc, "encoder")
    has_get_input_emb = hasattr(enc, "get_input_embeddings") and callable(enc.get_input_embeddings)
    accepts_inputs_embeds = "inputs_embeds" in enc.forward.__code__.co_varnames

    print(f"  Hidden dim: {H}")
    print(f"  Layers:     {NL} ({NL+1} hidden states with embedding layer)")
    print(f"  Params:     {n_params:.0f}M")
    print(f"  .embeddings attr:        {'yes' if has_emb_attr else 'NO'}")
    print(f"  .encoder attr:           {'yes' if has_encoder_attr else 'NO'}")
    print(f"  .get_input_embeddings(): {'yes' if has_get_input_emb else 'NO'}")
    print(f"  accepts inputs_embeds=:  {'yes' if accepts_inputs_embeds else 'NO'}")

    # Forward sanity
    with torch.no_grad():
        toks = tok("The cat sat on the mat.", return_tensors="pt")
        try:
            out = enc(**toks, output_hidden_states=True)
            print(f"  Forward OK — last hidden: {out.last_hidden_state.shape}, "
                  f"hidden_states tuple: {len(out.hidden_states)} layers")
        except Exception as e:
            print(f"  ✗ Forward failed: {type(e).__name__}: {e}")
            continue

        # Try predicate injection via get_input_embeddings + inputs_embeds
        if has_get_input_emb and accepts_inputs_embeds:
            try:
                emb_layer = enc.get_input_embeddings()
                emb = emb_layer(toks["input_ids"])
                # add a dummy "predicate" perturbation at first token
                emb = emb + torch.zeros_like(emb)
                out2 = enc(inputs_embeds=emb,
                           attention_mask=toks.get("attention_mask"),
                           output_hidden_states=True)
                print(f"  Predicate-injection path (inputs_embeds=) WORKS")
            except Exception as e:
                print(f"  ✗ inputs_embeds path failed: {type(e).__name__}: {e}")

    del enc, tok
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n" + "=" * 70)
print("If both encoders show 'inputs_embeds=' WORKS, proceed to Cell 15")
print("which defines an encoder-agnostic StudentCascadeUniversal that supports")
print("ModernBERT, NeoBERT, and DeBERTa from a single class.")
print("=" * 70)

ENCODER COMPATIBILITY CHECK

--- ModernBERT-base (149M) (answerdotai/ModernBERT-base) ---


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Hidden dim: 768
  Layers:     22 (23 hidden states with embedding layer)
  Params:     149M
  .embeddings attr:        yes
  .encoder attr:           NO
  .get_input_embeddings(): yes
  accepts inputs_embeds=:  NO
  Forward OK — last hidden: torch.Size([1, 9, 768]), hidden_states tuple: 23 layers

--- NeoBERT (250M) (chandar-lab/NeoBERT) ---


rotary.py: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/981M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] NeoBERT LOAD REPORT from: chandar-lab/NeoBERT
Key            | Status     |  | 
---------------+------------+--+-
decoder.bias   | UNEXPECTED |  | 
decoder.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Hidden dim: 768
  Layers:     28 (29 hidden states with embedding layer)
  Params:     222M
  .embeddings attr:        NO
  .encoder attr:           yes
  .get_input_embeddings(): yes
  accepts inputs_embeds=:  yes
  Forward OK — last hidden: torch.Size([1, 9, 768]), hidden_states tuple: 28 layers
  ✗ inputs_embeds path failed: NotImplementedError: `get_input_embeddings` not auto‑handled for NeoBERT; please override in the subclass.

If both encoders show 'inputs_embeds=' WORKS, proceed to Cell 15
which defines an encoder-agnostic StudentCascadeUniversal that supports
ModernBERT, NeoBERT, and DeBERTa from a single class.
